# Setup

In [ ]:
pip install rpy2

In [ ]:
%load_ext rpy2.ipython

## Install R Packages

In [ ]:
%%R
install.packages("gtsummary")

In [ ]:
%%R
install.packages("survival")
install.packages("survminer")
install.packages("lubridate")
install.packages("anytime")
install.packages("DataExplorer")
install.packages("patchwork")
install.packages("MatchIt")

In [ ]:
%%R
install.packages("caret")

In [ ]:
%%R
install.packages("gt")
install.packages("gtExtras")

In [ ]:
%%R
install.packages("pROC")

In [ ]:
%%R
install.packages("grid")
install.packages("gridExtra")

In [ ]:
%%R
install.packages("dcurves")

## Load R + Python Packages

In [ ]:
%%R
library(tidyverse)
library(survival)
library(survminer)
library(anytime)
library(lubridate)
library(DataExplorer)
library(patchwork)
library(pROC)
library(grid)
library(gridExtra)
library(gt)
library(gtExtras)
library(caret)
library(MatchIt)
library(gtsummary)
library(dcurves)

In [ ]:
import os
import numpy as np
import pandas as pd
import pandas_profiling
import subprocess
import plotnine
import matplotlib.pyplot as plt
from plotnine import *
from IPython.display import display
from datetime import datetime

## Plot setup.
theme_set(theme_bw(base_size = 11))

def get_boxplot_fun_data(df):
  """Returns a data frame with a y position and a label, for use annotating ggplot boxplots."""
  d = {'y': max(df), 'label': f'N = {len(df)}'}
  return(pd.DataFrame(data=d, index=[0]))

In [ ]:
import os
bucket = os.getenv("WORKSPACE_BUCKET")
bucket

# Import Kidney Stone Data

In [ ]:
import pandas
import os

# This query represents dataset "Kidney stones - symptomatic occurrence >6 months" for domain "condition" and was generated for All of Us Controlled Tier Dataset v8
# NOTE: BigQuery SQL queries are dataset-specific and generated by the AoU Cohort Builder.
# If using a different dataset version, regenerate the SQL from the AoU Researcher Workbench.
dataset_43325964_condition_sql = '''
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.condition_type_concept_id,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name
    FROM
        ( SELECT
            *
        FROM
            `''' + os.environ["WORKSPACE_CDR"] + '''.condition_occurrence` c_occurrence
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id
                FROM
                    `''' + os.environ["WORKSPACE_CDR"] + '''.cb_criteria` c
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id
                    FROM
                        `''' + os.environ["WORKSPACE_CDR"] + '''.cb_criteria` cr
                    WHERE
                        concept_id IN (201690, 201916, 4066163)
                        AND full_text LIKE '%_rank1]%'
                    ) a
                        ON (c.path LIKE CONCAT('%.', a.id, '.%')
                        OR c.path LIKE CONCAT('%.', a.id)
                        OR c.path LIKE CONCAT(a.id, '.%')
                        OR c.path = a.id)
                WHERE
                    is_standard = 1
                    AND is_selectable = 1)
                OR
                condition_source_concept_id IN (1571487, 35209253, 35209281, 35209282, 35209283, 35209284, 35209290, 44825543, 44826732, 44831593, 44833664, 44837195)
            )
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id
                FROM
                    `''' + os.environ["WORKSPACE_CDR"] + '''.cb_search_person` cb_search_person
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id
                        FROM
                            `''' + os.environ["WORKSPACE_CDR"] + '''.cb_search_all_events`
                        WHERE
                            (
                                concept_id IN (SELECT
                                    DISTINCT c.concept_id
                                FROM
                                    `''' + os.environ["WORKSPACE_CDR"] + '''.cb_criteria` c
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id
                                    FROM
                                        `''' + os.environ["WORKSPACE_CDR"] + '''.cb_criteria` cr
                                    WHERE
                                        concept_id IN (4160571, 4218106, 4080048, 4107882, 4058702, 4293099, 4163735, 4297872, 4232693, 4095693, 37110330, 4176173, 4163736, 4061849)
                                        AND full_text LIKE '%_rank1]%'
                                    ) a
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%')
                                    OR c.path LIKE CONCAT('%.', a.id)
                                    OR c.path LIKE CONCAT(a.id, '.%')
                                    OR c.path = a.id)
                                WHERE
                                    is_standard = 1
                                    AND is_selectable = 1)
                                OR
                                concept_id IN (SELECT
                                    DISTINCT c.concept_id
                                FROM
                                    `''' + os.environ["WORKSPACE_CDR"] + '''.cb_criteria` c
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id
                                    FROM
                                        `''' + os.environ["WORKSPACE_CDR"] + '''.cb_criteria` cr
                                    WHERE
                                        concept_id IN (201690, 201916, 4066163)
                                        AND full_text LIKE '%_rank1]%'
                                    ) a
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%')
                                    OR c.path LIKE CONCAT('%.', a.id)
                                    OR c.path LIKE CONCAT(a.id, '.%')
                                    OR c.path = a.id)
                                WHERE
                                    is_standard = 1
                                    AND is_selectable = 1)
                            )
                        ) criteria
                    )
                    AND cb_search_person.person_id IN (SELECT
                        person_id
                    FROM
                        `''' + os.environ["WORKSPACE_CDR"] + '''.cb_search_person` p
                    WHERE
                        has_whole_genome_variant = 1
                    )
                )
            )
        ) c_occurrence
    LEFT JOIN
        `''' + os.environ["WORKSPACE_CDR"] + '''.concept` c_standard_concept
            ON c_occurrence.condition_concept_id = c_standard_concept.concept_id
    LEFT JOIN
        `''' + os.environ["WORKSPACE_CDR"] + '''.concept` c_type
            ON c_occurrence.condition_type_concept_id = c_type.concept_id
    LEFT JOIN
        `''' + os.environ["WORKSPACE_CDR"] + '''.visit_occurrence` v
            ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id
    LEFT JOIN
        `''' + os.environ["WORKSPACE_CDR"] + '''.concept` visit
            ON v.visit_concept_id = visit.concept_id
    LEFT JOIN
        `''' + os.environ["WORKSPACE_CDR"] + '''.concept` c_source_concept
            ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id
    LEFT JOIN
        `''' + os.environ["WORKSPACE_CDR"] + '''.concept` c_status
            ON c_occurrence.condition_status_concept_id = c_status.concept_id
'''

symptomatic_occurrence_diagnostic_codes = pandas.read_gbq(
    dataset_43325964_condition_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

symptomatic_occurrence_diagnostic_codes.head(10)

In [ ]:
import pandas
import os

# This query represents dataset "Kidney stones - symptomatic occurrence >6 months" for domain "procedure" and was generated for All of Us Controlled Tier Dataset v8
dataset_43325964_procedure_sql = """
    SELECT
        procedure.person_id,
        procedure.procedure_concept_id,
        p_standard_concept.concept_name as standard_concept_name,
        p_standard_concept.concept_code as standard_concept_code,
        p_standard_concept.vocabulary_id as standard_vocabulary,
        procedure.procedure_datetime,
        procedure.procedure_type_concept_id,
        p_type.concept_name as procedure_type_concept_name,
        procedure.modifier_concept_id,
        p_modifier.concept_name as modifier_concept_name,
        procedure.quantity,
        procedure.visit_occurrence_id,
        p_visit.concept_name as visit_occurrence_concept_name,
        procedure.procedure_source_value,
        procedure.procedure_source_concept_id,
        p_source_concept.concept_name as source_concept_name,
        p_source_concept.concept_code as source_concept_code,
        p_source_concept.vocabulary_id as source_vocabulary,
        procedure.modifier_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.procedure_occurrence` procedure 
        WHERE
            (
                procedure_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (2721096, 4087889, 4121225, 4170611, 4171381, 4237124, 4301429, 4341377)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1) 
                OR  procedure_source_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (2003682, 2008219, 2008220, 2008221, 2109553, 2109556, 2109557, 2109558, 2109563, 2109626, 2109634, 2109635, 2109641, 2109687, 2109803, 2109804, 2109816, 2109817, 2721096, 2774214, 2774218, 2774219, 2774220, 2774224, 44816428)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 0 
                    AND is_selectable = 1)
            )  
            AND (
                procedure.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            person_id IN (SELECT
                                person_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                            WHERE
                                concept_id IN(SELECT
                                    DISTINCT c.concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id       
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                    WHERE
                                        concept_id IN (4237124, 4121225, 4087889, 4301429, 2721096, 4341377, 4170611, 4171381)       
                                        AND full_text LIKE '%_rank1]%'      ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) 
                                AND is_standard = 1 
                            UNION
                            DISTINCT SELECT
                                person_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                            WHERE
                                concept_id IN(SELECT
                                    DISTINCT c.concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id       
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                    WHERE
                                        concept_id IN (2109816, 2109804, 2109554, 2109556, 2109803, 2774224, 2109687, 2832813, 2109557, 2109563, 2109553, 2824569, 2109634, 2774220, 2109635, 2008219, 2008218, 2721096, 2819509, 2109626, 2109817, 2109641, 2003682, 2109558, 44816428)       
                                        AND full_text LIKE '%_rank1]%'      ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 0 
                                    AND is_selectable = 1) 
                                AND is_standard = 0 )) criteria 
                        UNION
                        DISTINCT SELECT
                            criteria.person_id 
                        FROM
                            (SELECT
                                DISTINCT person_id, entry_date, concept_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                            WHERE
                                person_id IN (SELECT
                                    person_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    concept_id IN(SELECT
                                        DISTINCT c.concept_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                                    JOIN
                                        (SELECT
                                            CAST(cr.id as string) AS id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                        WHERE
                                            concept_id IN (201690, 4066163, 201916)       
                                            AND full_text LIKE '%_rank1]%'      ) a 
                                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                            OR c.path LIKE CONCAT('%.', a.id) 
                                            OR c.path LIKE CONCAT(a.id, '.%') 
                                            OR c.path = a.id) 
                                    WHERE
                                        is_standard = 1 
                                        AND is_selectable = 1) 
                                    AND is_standard = 1 
                                UNION
                                DISTINCT SELECT
                                    person_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    concept_id IN(SELECT
                                        DISTINCT c.concept_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                                    JOIN
                                        (SELECT
                                            CAST(cr.id as string) AS id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                        WHERE
                                            concept_id IN (1571487, 44831593, 44837195, 35209290, 35209253)       
                                            AND full_text LIKE '%_rank1]%'      ) a 
                                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                            OR c.path LIKE CONCAT('%.', a.id) 
                                            OR c.path LIKE CONCAT(a.id, '.%') 
                                            OR c.path = a.id) 
                                    WHERE
                                        is_standard = 0 
                                        AND is_selectable = 1) 
                                    AND is_standard = 0 )) criteria ) 
                                AND cb_search_person.person_id IN (SELECT
                                    person_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                                WHERE
                                    has_whole_genome_variant = 1 ) )
                            )
                        ) procedure 
                    LEFT JOIN
                        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_standard_concept 
                            ON procedure.procedure_concept_id = p_standard_concept.concept_id 
                    LEFT JOIN
                        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_type 
                            ON procedure.procedure_type_concept_id = p_type.concept_id 
                    LEFT JOIN
                        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_modifier 
                            ON procedure.modifier_concept_id = p_modifier.concept_id 
                    LEFT JOIN
                        `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
                            ON procedure.visit_occurrence_id = v.visit_occurrence_id 
                    LEFT JOIN
                        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_visit 
                            ON v.visit_concept_id = p_visit.concept_id 
                    LEFT JOIN
                        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_source_concept 
                            ON procedure.procedure_source_concept_id = p_source_concept.concept_id"""

symptomatic_occurrence_procedure_codes = pandas.read_gbq(
    dataset_43325964_procedure_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

symptomatic_occurrence_procedure_codes.head(10)

In [ ]:
import pandas
import os

# This query represents dataset "Kidney stones - all patients" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
# Returns person_id, gender, date_of_birth for kidney stone patients with WGS
dataset_78141828_person_sql = '''
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth
    FROM
        `''' + os.environ["WORKSPACE_CDR"] + '''.person` person
    LEFT JOIN
        `''' + os.environ["WORKSPACE_CDR"] + '''.concept` p_gender_concept
            ON person.gender_concept_id = p_gender_concept.concept_id
    LEFT JOIN
        `''' + os.environ["WORKSPACE_CDR"] + '''.concept` p_race_concept
            ON person.race_concept_id = p_race_concept.concept_id
    LEFT JOIN
        `''' + os.environ["WORKSPACE_CDR"] + '''.concept` p_ethnicity_concept
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id
    LEFT JOIN
        `''' + os.environ["WORKSPACE_CDR"] + '''.concept` p_sex_at_birth_concept
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id
        FROM
            `''' + os.environ["WORKSPACE_CDR"] + '''.cb_search_person` cb_search_person
        WHERE
            cb_search_person.person_id IN (SELECT
                criteria.person_id
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id
                FROM
                    `''' + os.environ["WORKSPACE_CDR"] + '''.cb_search_all_events`
                WHERE
                    (
                        concept_id IN (SELECT
                            DISTINCT c.concept_id
                        FROM
                            `''' + os.environ["WORKSPACE_CDR"] + '''.cb_criteria` c
                        JOIN
                            (SELECT
                                CAST(cr.id as string) AS id
                            FROM
                                `''' + os.environ["WORKSPACE_CDR"] + '''.cb_criteria` cr
                            WHERE
                                concept_id IN (4160571, 4218106, 4080048, 4107882, 4058702, 4293099, 4163735, 4297872, 4232693, 4095693, 37110330, 4176173, 4163736, 4061849)
                                AND full_text LIKE '%_rank1]%'
                            ) a
                            ON (c.path LIKE CONCAT('%.', a.id, '.%')
                            OR c.path LIKE CONCAT('%.', a.id)
                            OR c.path LIKE CONCAT(a.id, '.%')
                            OR c.path = a.id)
                        WHERE
                            is_standard = 1
                            AND is_selectable = 1)
                        OR
                        concept_id IN (SELECT
                            DISTINCT c.concept_id
                        FROM
                            `''' + os.environ["WORKSPACE_CDR"] + '''.cb_criteria` c
                        JOIN
                            (SELECT
                                CAST(cr.id as string) AS id
                            FROM
                                `''' + os.environ["WORKSPACE_CDR"] + '''.cb_criteria` cr
                            WHERE
                                concept_id IN (201690, 201916, 4066163)
                                AND full_text LIKE '%_rank1]%'
                            ) a
                            ON (c.path LIKE CONCAT('%.', a.id, '.%')
                            OR c.path LIKE CONCAT('%.', a.id)
                            OR c.path LIKE CONCAT(a.id, '.%')
                            OR c.path = a.id)
                        WHERE
                            is_standard = 1
                            AND is_selectable = 1)
                    )
                ) criteria
            )
            AND cb_search_person.person_id IN (SELECT
                person_id
            FROM
                `''' + os.environ["WORKSPACE_CDR"] + '''.cb_search_person` p
            WHERE
                has_whole_genome_variant = 1
            )
        )
'''

overall_person_df = pandas.read_gbq(
    dataset_78141828_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

overall_person_df.head(10)

In [ ]:
import pandas
import os

# This query represents dataset "Kidney stones - all patients ethnicity" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
# Returns person demographics including race/ethnicity
dataset_28927342_person_sql = '''
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth
    FROM
        `''' + os.environ["WORKSPACE_CDR"] + '''.person` person
    LEFT JOIN
        `''' + os.environ["WORKSPACE_CDR"] + '''.concept` p_gender_concept
            ON person.gender_concept_id = p_gender_concept.concept_id
    LEFT JOIN
        `''' + os.environ["WORKSPACE_CDR"] + '''.concept` p_race_concept
            ON person.race_concept_id = p_race_concept.concept_id
    LEFT JOIN
        `''' + os.environ["WORKSPACE_CDR"] + '''.concept` p_ethnicity_concept
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id
    LEFT JOIN
        `''' + os.environ["WORKSPACE_CDR"] + '''.concept` p_sex_at_birth_concept
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id
        FROM
            `''' + os.environ["WORKSPACE_CDR"] + '''.cb_search_person` cb_search_person
        WHERE
            cb_search_person.person_id IN (SELECT
                criteria.person_id
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id
                FROM
                    `''' + os.environ["WORKSPACE_CDR"] + '''.cb_search_all_events`
                WHERE
                    (
                        concept_id IN (SELECT
                            DISTINCT c.concept_id
                        FROM
                            `''' + os.environ["WORKSPACE_CDR"] + '''.cb_criteria` c
                        JOIN
                            (SELECT
                                CAST(cr.id as string) AS id
                            FROM
                                `''' + os.environ["WORKSPACE_CDR"] + '''.cb_criteria` cr
                            WHERE
                                concept_id IN (4160571, 4218106, 4080048, 4107882, 4058702, 4293099, 4163735, 4297872, 4232693, 4095693, 37110330, 4176173, 4163736, 4061849)
                                AND full_text LIKE '%_rank1]%'
                            ) a
                            ON (c.path LIKE CONCAT('%.', a.id, '.%')
                            OR c.path LIKE CONCAT('%.', a.id)
                            OR c.path LIKE CONCAT(a.id, '.%')
                            OR c.path = a.id)
                        WHERE
                            is_standard = 1
                            AND is_selectable = 1)
                        OR
                        concept_id IN (SELECT
                            DISTINCT c.concept_id
                        FROM
                            `''' + os.environ["WORKSPACE_CDR"] + '''.cb_criteria` c
                        JOIN
                            (SELECT
                                CAST(cr.id as string) AS id
                            FROM
                                `''' + os.environ["WORKSPACE_CDR"] + '''.cb_criteria` cr
                            WHERE
                                concept_id IN (201690, 201916, 4066163)
                                AND full_text LIKE '%_rank1]%'
                            ) a
                            ON (c.path LIKE CONCAT('%.', a.id, '.%')
                            OR c.path LIKE CONCAT('%.', a.id)
                            OR c.path LIKE CONCAT(a.id, '.%')
                            OR c.path = a.id)
                        WHERE
                            is_standard = 1
                            AND is_selectable = 1)
                    )
                ) criteria
            )
            AND cb_search_person.person_id IN (SELECT
                person_id
            FROM
                `''' + os.environ["WORKSPACE_CDR"] + '''.cb_search_person` p
            WHERE
                has_whole_genome_variant = 1
            )
        )
'''

overall_person_df_ethnicity = pandas.read_gbq(
    dataset_28927342_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

overall_person_df_ethnicity.head(5)

In [ ]:
import pandas
import os

# This query represents dataset "Final Dates" for domain "condition" and was generated for All of Us Controlled Tier Dataset v8
dataset_88903298_condition_sql = """
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.condition_type_concept_id,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (133468, 134057, 134736, 135930, 138239, 138525, 140190, 141960, 192438, 193460, 194133, 200174, 200219, 200452, 201618, 201891, 253549, 254068, 254761, 255919, 31610, 316866, 31821, 318800, 320128, 320136, 321588, 37018424, 37018713, 37203927, 37206233, 372887, 37311677, 37311678, 373499, 375252, 376106, 376208, 376337, 4000609, 4000610, 4002898, 4002905, 4006969, 4009890, 4011630, 4020346, 4021667, 4021780, 4021915, 4022201, 4022922, 4022923, 4022924, 4023995, 4024000, 4024013, 4024561, 4024566, 4024567, 4025215, 4027384, 4027553, 4028071, 4028076, 4028387, 4031814, 4038502, 4038677, 4038678, 4041283, 4041284, 4041285, 4041436, 4042056, 4042140, 4042141, 4042142, 4042503, 4042505, 4042835, 4042836, 4042837, 4043346, 4043371, 4043671, 4047779, 40480457, 40481517, 40481841, 40484102, 40484533, 40484935, 4051221, 4054501, 4071689, 4080992, 4081176, 4083787, 4086181, 4090426, 4090614, 4090615, 4091213, 4091363, 4091532, 4093227, 4093228, 4093991,
 4094294, 4095793, 4096864, 4100932, 4101212, 4101343, 4101796, 4102111, 4103183, 4103320, 4103352, 4104157, 4113563, 4113999, 4115259, 4115386, 4115390, 4116811, 4117779, 4117930, 4132555, 4132926, 4134294, 4134440, 4150129, 4160062, 4162282, 4168335, 4170143, 4170226, 4170962, 4171379, 4175154, 4178545, 4178680, 4178818, 4179141, 4179167, 4179871, 4179873, 4180154, 4180169, 4180170, 4180628, 4181063, 4181187, 4182161, 4182165, 4182633, 4184252, 4185503, 4185640, 4190185, 4197094, 4198525, 4199402, 4200532, 4201745, 4206591, 4208786, 4212577, 4213101, 4221108, 4227253, 4244662, 4247371, 42538830, 4260918, 4266186, 4267789, 4269314, 4272867, 4274025, 4276569, 4276572, 4277352, 4293175, 4297887, 4301699, 4302537, 4304916, 4305027, 4308811, 4309188, 4316083, 4317258, 4318379, 432250, 432586, 432795, 432867, 4329041, 433128, 433736, 4338120, 4339410, 4339468, 4344497, 43530815, 43530877, 43531053, 43531054, 43531056, 43531057, 43531058, 43531059, 43531060, 435506, 435524, 436096, 436670,
 437515, 438112, 440029, 440142, 440371, 440383, 440921, 441542, 442077, 443343, 443723, 443783, 443784, 443883, 444089, 444100, 444108, 444112, 444363, 44782620, 44783587, 44784102, 73553, 75865, 75909, 77074, 77670, 77960, 79106, 80180)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT c.concept_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                WHERE
                                    concept_id IN (40480457, 4042140, 4011630, 4274025, 4168335, 4094294, 4021667)       
                                    AND full_text LIKE '%_rank1]%'      ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) 
                            AND is_standard = 1 )) criteria 
                    UNION
                    DISTINCT SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT c.concept_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                WHERE
                                    concept_id IN (2617204, 4029205, 704057, 4180627, 2617208, 1314331, 2720580, 4237321, 4180941, 40295754, 40217302, 4302541, 4132623, 45890623, 4028908, 4072499, 2721530, 4237320, 4176642)       
                                    AND full_text LIKE '%_rank1]%'      ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) 
                            AND is_standard = 1 )) criteria ) )
                )
            ) c_occurrence 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_standard_concept 
                ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_type 
                ON c_occurrence.condition_type_concept_id = c_type.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
                ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` visit 
                ON v.visit_concept_id = visit.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_source_concept 
                ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_status 
                ON c_occurrence.condition_status_concept_id = c_status.concept_id"""

overall_condition_df = pandas.read_gbq(
    dataset_88903298_condition_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

overall_condition_df.head(5)

In [ ]:
import pandas
import os

# This query represents dataset "Final Dates" for domain "procedure" and was generated for All of Us Controlled Tier Dataset v8
dataset_88903298_procedure_sql = """
    SELECT
        procedure.person_id,
        procedure.procedure_concept_id,
        p_standard_concept.concept_name as standard_concept_name,
        p_standard_concept.concept_code as standard_concept_code,
        p_standard_concept.vocabulary_id as standard_vocabulary,
        procedure.procedure_datetime,
        procedure.procedure_type_concept_id,
        p_type.concept_name as procedure_type_concept_name,
        procedure.modifier_concept_id,
        p_modifier.concept_name as modifier_concept_name,
        procedure.quantity,
        procedure.visit_occurrence_id,
        p_visit.concept_name as visit_occurrence_concept_name,
        procedure.procedure_source_value,
        procedure.procedure_source_concept_id,
        p_source_concept.concept_name as source_concept_name,
        p_source_concept.concept_code as source_concept_code,
        p_source_concept.vocabulary_id as source_vocabulary,
        procedure.modifier_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.procedure_occurrence` procedure 
        WHERE
            (
                procedure_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (1314331, 2617204, 2617208, 2617223, 2720580, 2721530, 36675054, 36714147, 37204303, 37204304, 4000738, 4001217, 4002365, 4002733, 4003041, 4004516, 4010109, 4012185, 4012907, 4013636, 4016388, 4017459, 40217302, 4024612, 4027403, 4028902, 4028908, 4028909, 4029205, 40295754, 4029698, 4030028, 4030654, 4030660, 4031967, 4032237, 4032251, 4032254, 4032257, 4036803, 4037672, 4039376, 4040390, 4040391, 4040392, 4040393, 4040395, 4040396, 4040400, 4040551, 4040556, 4040559, 4040561, 4040563, 4040721, 4041261, 4041263, 4041982, 4042150, 4042330, 4042486, 4042493, 4042504, 4042516, 4042641, 4042642, 4042645, 4042646, 4042650, 4042660, 4042673, 4042677, 4042815, 4043017, 4043018, 4043019, 4043022, 4043023, 4043025, 4043028, 4043161, 4043176, 4043178, 4043179, 4043182, 4043197, 4043334, 4043336, 4044012, 4044875, 4044887, 4044900, 4045893, 4045900, 4047347, 40480674, 40480925, 40481382, 40481384, 40481878, 40482428, 4048375, 40487842, 4052492, 4063521,
 4064522, 4065062, 4067285, 4072499, 4073050, 4073623, 4074287, 4075084, 4075966, 4077953, 4078442, 4078460, 4079711, 4088217, 4093436, 4100357, 4104795, 4105895, 4108616, 4109665, 4111522, 4111529, 4111534, 4117767, 4118601, 4121315, 4124406, 4125501, 4128030, 4128032, 4129747, 4132623, 4132648, 4132855, 4133169, 4133312, 4134148, 4134423, 4134598, 4135174, 4138127, 4139262, 4141759, 4144375, 4145308, 4147894, 4155790, 4155793, 4158570, 4159949, 4160360, 4160550, 4162054, 4163951, 4164278, 4168709, 4172515, 4172654, 4176642, 4176720, 4177094, 4178367, 4179713, 4180004, 4180627, 4180642, 4180653, 4180938, 4180941, 4181192, 4181193, 4181322, 4181638, 4181966, 4185115, 4185623, 4189705, 4190496, 4190759, 4194672, 4195757, 4196582, 4196958, 4201944, 4202517, 4202832, 4208079, 4211641, 4213297, 4214115, 4216752, 4221994, 4230374, 4230911, 4231758, 4233946, 4237320, 4237321, 4239679, 4240345, 4241075, 4243910, 4247168, 4248675, 4249893, 4250598, 42536500, 4254477, 4254480, 4258374, 4261206,
 4263508, 4264477, 4266234, 4268628, 4270484, 4279903, 4287856, 4293032, 4293902, 4297090, 4297249, 4299523, 4299602, 4300244, 4300757, 4301351, 4302532, 4302541, 4304358, 4306074, 4306780, 4310912, 4311405, 4313306, 4313889, 4314450, 4324693, 4326426, 43531071, 43531072, 45773385, 45890623, 46272569, 704057)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                procedure.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT c.concept_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                WHERE
                                    concept_id IN (40480457, 4042140, 4011630, 4274025, 4168335, 4094294, 4021667)       
                                    AND full_text LIKE '%_rank1]%'      ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) 
                            AND is_standard = 1 )) criteria 
                    UNION
                    DISTINCT SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT c.concept_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                WHERE
                                    concept_id IN (2617204, 4029205, 704057, 4180627, 2617208, 1314331, 2720580, 4237321, 4180941, 40295754, 40217302, 4302541, 4132623, 45890623, 4028908, 4072499, 2721530, 4237320, 4176642)       
                                    AND full_text LIKE '%_rank1]%'      ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) 
                            AND is_standard = 1 )) criteria ) )
                )
            ) procedure 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_standard_concept 
                ON procedure.procedure_concept_id = p_standard_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_type 
                ON procedure.procedure_type_concept_id = p_type.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_modifier 
                ON procedure.modifier_concept_id = p_modifier.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
                ON procedure.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_visit 
                ON v.visit_concept_id = p_visit.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_source_concept 
                ON procedure.procedure_source_concept_id = p_source_concept.concept_id"""


overall_procedures_df = pandas.read_gbq(
    dataset_88903298_procedure_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

overall_procedures_df.head(5)

In [ ]:
ethnicity_factors = overall_person_df_ethnicity['race'].unique()
print(f'Different factors in the ethnicity column: {ethnicity_factors}')

total_count = overall_person_df_ethnicity.shape[0]
white_count = overall_person_df_ethnicity[overall_person_df_ethnicity['race'] == 'White'].shape[0]
white_proportion = white_count / total_count
print(f'Proportion of White participants: {white_proportion:.2%}')

In [ ]:
import pandas as pd

symp_proc_selected = symptomatic_occurrence_procedure_codes[['person_id', 'procedure_datetime']].rename(
    columns={'procedure_datetime': 'date_time'})
overall_procedure_selected = overall_procedures_df[['person_id', 'procedure_datetime']].rename(
    columns={'procedure_datetime': 'date_time'})
symp_diag_selected = symptomatic_occurrence_diagnostic_codes[['person_id', 'condition_start_datetime']].rename(
    columns={'condition_start_datetime': 'date_time'})
overall_condition_selected = overall_condition_df[['person_id', 'condition_start_datetime']].rename(
    columns={'condition_start_datetime': 'date_time'})

combined_overall_df = pd.concat([
    overall_condition_selected, overall_procedure_selected,
    symp_proc_selected, symp_diag_selected
], ignore_index=True)
combined_overall_df['date_time'] = pd.to_datetime(combined_overall_df['date_time'])

final_date_df = combined_overall_df.groupby('person_id', as_index=False).agg(
    final_date=('date_time', 'max'))
final_date_df.info()

# Merge Kidney Stone Datasets
## Sort Colic Dates

In [ ]:
%%R -i symptomatic_occurrence_diagnostic_codes

symptomatic_occurrence_diagnostic_codes <- as_tibble(symptomatic_occurrence_diagnostic_codes)
str(symptomatic_occurrence_diagnostic_codes)

In [ ]:
%%R
table(symptomatic_occurrence_diagnostic_codes$standard_concept_name)

In [ ]:
%%R

symptomatic_occurrence_diagnostic_codes <- symptomatic_occurrence_diagnostic_codes %>% mutate(
            initial_presentation = ifelse(
                standard_concept_name == "Calculus in pelviureteric junction" |
                standard_concept_name == "Calculus of kidney and ureter" |
                standard_concept_name == "Calculus of upper urinary tract" |
                standard_concept_name == "Hydronephrosis due to calculus of kidney and ureter" |
                standard_concept_name == "Renal colic" |
                standard_concept_name == "Ureteric colic" |
                standard_concept_name == "Ureteric stone of lower third of ureter" |
                standard_concept_name == "Ureteric stone" |
                standard_concept_name == "Urinary tract obstruction" |
                standard_concept_name == "Vesicoureteric junction calculus",
                "Colic",
                "Kidney_stone"
            )
)

In [ ]:
%%R
table(symptomatic_occurrence_diagnostic_codes$initial_presentation)

## Sort Interventions

In [ ]:
%%R -i symptomatic_occurrence_procedure_codes

symptomatic_occurrence_procedure_codes <- as_tibble(symptomatic_occurrence_procedure_codes)
str(symptomatic_occurrence_procedure_codes)

In [ ]:
%%R
table(symptomatic_occurrence_procedure_codes$standard_concept_name)

In [ ]:
%%R

symptomatic_occurrence_procedure_codes <- symptomatic_occurrence_procedure_codes %>% mutate(
            initial_presentation = ifelse(standard_concept_name == "Cystourethroscopy, with ureteroscopy and/or pyeloscopy (includes dilation of the ureter and/or pyeloureteral junction by any method); with lithotripsy (ureteral catheterization is included) (Deprecated)",
                                          "URS",
            ifelse(standard_concept_name == "Cystourethroscopy, with ureteroscopy and/or pyeloscopy (includes dilation of the ureter and/or pyeloureteral junction by any method); with removal or manipulation of calculus (ureteral catheterization is included) (Deprecated)",
                   "URS",
            ifelse(standard_concept_name == "Cystourethroscopy, with ureteroscopy and/or pyeloscopy; with lithotripsy including insertion of indwelling ureteral stent (eg, Gibbons or double-J type)",
                   "URS",
            ifelse(standard_concept_name == "Cystourethroscopy, with ureteroscopy and/or pyeloscopy; with removal or manipulation of calculus (ureteral catheterization is included)",
                   "URS",
            ifelse(standard_concept_name == "Extracorporeal shockwave lithotripsy",
                  "ESWL",
            ifelse(standard_concept_name == "Extracorporeal shockwave lithotripsy [ESWL] of the kidney, ureter and/or bladder",
                   "ESWL",
            ifelse(standard_concept_name == "Extracorporeal shockwave lithotripsy of calculus of kidney",
                   "ESWL",
            ifelse(standard_concept_name == "Extracorporeal shockwave lithotripsy of other sites",
                   "ESWL",
            ifelse(standard_concept_name == "Fragmentation in Left Ureter, Open Approach",
                  "Open",
            ifelse(standard_concept_name == "Fragmentation in Left Ureter, Via Natural or Artificial Opening Endoscopic",
                   "URS",
            ifelse(standard_concept_name == "Fragmentation in Right Ureter, External Approach",
                   "Open",
            ifelse(standard_concept_name == "Fragmentation in Right Ureter, Via Natural or Artificial Opening Endoscopic",
                   "URS",
            ifelse(standard_concept_name == "Nephrolithotomy for removal of calculus",
                   "PCNL",
            ifelse(standard_concept_name == "Nephrolithotomy for removal of staghorn calculus ",
                  "PCNL",
            ifelse(standard_concept_name == "Percutaneous extraction of kidney stone with fragmentation procedure",
                   "PCNL",
            ifelse(standard_concept_name == "Percutaneous nephrolithotomy or pyelolithotomy, lithotripsy, stone extraction, antegrade ureteroscopy, antegrade stent placement and nephrostomy tube placement, when performed, including imaging guidance; complex (eg, stone[s] > 2 cm, branching stones, st",
                   "PCNL",
            ifelse(standard_concept_name == "Percutaneous nephrolithotomy or pyelolithotomy, lithotripsy, stone extraction, antegrade ureteroscopy, antegrade stent placement and nephrostomy tube placement, when performed, including imaging guidance; simple (eg, stone[s] up to 2 cm in single location",
                   "PCNL",
            ifelse(standard_concept_name == "Pyelolithotomy",
                   "Open",
            ifelse(standard_concept_name == "Removal of calculus of renal pelvis through percutaneous nephrostomy",
                   "PCNL",
            ifelse(standard_concept_name == "Renal endoscopy through established nephrostomy or pyelostomy, with or without irrigation, instillation, or ureteropyelography, exclusive of radiologic service; with removal of foreign body or calculus",
                   "PCNL",
            ifelse(standard_concept_name == "Renal endoscopy through nephrotomy or pyelotomy, with or without irrigation, instillation, or ureteropyelography, exclusive of radiologic service; with removal of foreign body or calculus",
                   "PCNL",
            ifelse(standard_concept_name == "Renal endoscopy through nephrotomy or pyelotomy, with or without irrigation, instillation, or ureteropyelography, exclusive of radiologic service; with removal of foreign body or calculus",
                   "PCNL",
            ifelse(standard_concept_name == "Ureteral endoscopy through established ureterostomy, with or without irrigation, instillation, or ureteropyelography, exclusive of radiologic service; with removal of foreign body or calculus",
                    "URS",
            ifelse(standard_concept_name == "Ureterolithotomy, upper one-third of ureter",
                   "Open",
            ifelse(standard_concept_name == "Ureteroscopic laser fragmentation of ureteric calculus",
                   "URS",
            ifelse(standard_concept_name == "Ureteroscopy",
                   "URS",
            ifelse(standard_concept_name == "Ureteroscopy with biopsy ",
                   "URS",
                   NA
            )))))))))))))))))))))))))))
)

In [ ]:
%%R
symptomatic_occurrence_procedure_codes_only <- symptomatic_occurrence_procedure_codes %>% group_by(person_id)
str(symptomatic_occurrence_procedure_codes_only)

In [ ]:
symptomatic_occurrence_diagnostic_codes = %Rget symptomatic_occurrence_diagnostic_codes
symptomatic_occurrence_procedure_codes = %Rget symptomatic_occurrence_procedure_codes

In [ ]:
diagnostic_selected = symptomatic_occurrence_diagnostic_codes[['person_id', 'condition_start_datetime', 'initial_presentation']]
diagnostic_selected = diagnostic_selected.dropna(subset=['initial_presentation'])
diagnostic_selected = diagnostic_selected.rename(columns={'condition_start_datetime': 'date_time'})

procedure_selected = symptomatic_occurrence_procedure_codes[['person_id', 'procedure_datetime', 'initial_presentation']]
procedure_selected = procedure_selected.dropna(subset=['initial_presentation'])
procedure_selected = procedure_selected.rename(columns={'procedure_datetime': 'date_time'})

combined_df = pd.concat([diagnostic_selected, procedure_selected], ignore_index=True)
combined_df.info()

In [ ]:
%%R -i combined_df
query_combined_df <- as_tibble(combined_df)
query_combined_df$date_time <- anydate(query_combined_df$date_time)
str(query_combined_df)

In [ ]:
%%R
intervention_pattern <- "PCNL|URS|ESWL|Open"

initial_intervention_df <- query_combined_df %>%
  filter(!grepl("Kidney_stone", initial_presentation)) %>%
  arrange(person_id, date_time) %>%
  group_by(person_id) %>%
  group_modify(~{
    df <- .x
    first_event <- df[1, ]

    if (grepl(intervention_pattern, first_event$initial_presentation)) {
      return(tibble(
        initial_presentation = first_event$initial_presentation,
        first_date = first_event$date_time
      ))
    }

    if (grepl("Colic", first_event$initial_presentation)) {
      intervention_within_6m <- df %>%
        filter(
          date_time > first_event$date_time,
          date_time <= first_event$date_time %m+% months(6),
          grepl(intervention_pattern, initial_presentation)
        ) %>%
        slice_min(date_time, n = 1)

      if (nrow(intervention_within_6m) > 0) {
        return(tibble(
          initial_presentation = first_event$initial_presentation,
          first_date = intervention_within_6m$date_time
        ))
      }

      return(tibble(
        initial_presentation = first_event$initial_presentation,
        first_date = first_event$date_time
      ))
    }

    return(tibble(
      initial_presentation = first_event$initial_presentation,
      first_date = first_event$date_time
    ))
  }) %>% ungroup()

table(initial_intervention_df$initial_presentation)

In [ ]:
%%R
str(initial_intervention_df)

In [ ]:
%%R

second_event_df <- initial_intervention_df %>%
  left_join(
    query_combined_df %>%
      filter(!grepl("Kidney_stone", initial_presentation)),
    by = "person_id"
  ) %>%
  filter(date_time >= first_date %m+% months(6)) %>%
  group_by(person_id) %>%
  summarise(
    second_date = min(date_time),
    second_event_flag = TRUE,
    .groups = "drop"
  )

final_df <- initial_intervention_df %>%
  left_join(second_event_df, by = "person_id") %>%
  mutate(
    second_event_flag = ifelse(is.na(second_event_flag), FALSE, second_event_flag)
  )

str(final_df)

In [ ]:
%%R
table(final_df$second_event_flag)

In [ ]:
%%R

clean_df <- query_combined_df %>%
  filter(!grepl("Kidney_stone", initial_presentation))

last_followup <- clean_df %>%
  group_by(person_id) %>%
  summarise(last_followup_date = max(date_time), .groups = "drop")

km_df <- initial_intervention_df %>%
  left_join(second_event_df, by = "person_id") %>%
  left_join(last_followup, by = "person_id") %>%
  mutate(
    first_date = as.Date(first_date),
    second_date = as.Date(second_date),
    last_followup_date = as.Date(last_followup_date),
    event = ifelse(!is.na(second_date), 1, 0),
    end_date = ifelse(event == 1, second_date, last_followup_date),
    end_date = as.Date(end_date),
    time_days = as.numeric(end_date - first_date)
  )

surv_object <- Surv(time = km_df$time_days, event = km_df$event)

km_fit <- survfit(
  Surv(time_days/365.25, event) ~ 1,
  data = km_df
)

ggsurvplot(
  km_fit,
  data = km_df,
  risk.table = TRUE,
  surv.median.line = "hv",
  conf.int = TRUE,
  break.time.by = 1,
  xlab = "Time (years)",
  ylab = "Survival probability",
  ggtheme = theme_bw(),
  xlim = c(0, 10),
  title = "Time to First Recurrence"
)

## Combine Colic + Interventions Dataset

In [ ]:
%%R

symptomatic_occurrence_procedure_codes1 <- symptomatic_occurrence_procedure_codes %>% subset(select = c(person_id,
                                                                                                         procedure_datetime,
                                                                                                         initial_presentation)) %>% mutate(
date_time = procedure_datetime, .keep = "unused")

symptomatic_occurrence_diagnostic_codes1 <- symptomatic_occurrence_diagnostic_codes %>% subset(select = c(person_id,
                                                                                                         condition_start_datetime,
                                                                                                         initial_presentation)) %>% mutate(
date_time = condition_start_datetime, .keep = "unused")

combined_data <- rbind(symptomatic_occurrence_procedure_codes1,
                       symptomatic_occurrence_diagnostic_codes1) %>% as_tibble()

combined_data <- combined_data %>% arrange(desc(date_time))

str(combined_data)

## Get Symptomatic Occurrences Dataset

In [ ]:
%%R

symptomatic_occurrence_procedure_codes2 <- symptomatic_occurrence_procedure_codes %>%
    subset(select = c(person_id,
                      procedure_datetime,
                      initial_presentation)) %>%
    mutate(
        date_time = procedure_datetime,
        .keep = "unused")

symptomatic_occurrence_diagnostic_codes2 <- symptomatic_occurrence_diagnostic_codes %>%
    subset(initial_presentation != "Kidney_stone") %>%
    subset(select = c(person_id,
                      condition_start_datetime,
                      initial_presentation)) %>%
    mutate(
        date_time = condition_start_datetime,
        .keep = "unused")

combined_data2 <- rbind(symptomatic_occurrence_procedure_codes2,
                       symptomatic_occurrence_diagnostic_codes2) %>% as_tibble()

combined_data2$date_time <- as.Date(combined_data2$date_time)

combined_data2 <- combined_data2 %>% arrange(desc(date_time))

str(combined_data2)

In [ ]:
%%R
n_distinct(combined_data2$person_id)

In [ ]:
%%R -i overall_person_df_ethnicity

age_at_first_stone_df_1 <- as_tibble(overall_person_df_ethnicity) %>%
                            dplyr::select(person_id, date_of_birth)

str(age_at_first_stone_df_1)

In [ ]:
%%R
n_distinct(combined_data$person_id)

In [ ]:
%%R
n_distinct(combined_data2$person_id)

In [ ]:
%%R

symptomatic_episodes_data <- combined_data2 %>%
  distinct(person_id, date_time, initial_presentation) %>%
  mutate(initial_presentation = case_when(
      is.na(initial_presentation) ~ "Colic",
      TRUE ~ initial_presentation
  )) %>%
  group_by(person_id) %>%
  summarise(
    symptomatic_episode_date = list(sort(date_time)),
    initial_presentation = list(initial_presentation[order(date_time)])
  )

str(symptomatic_episodes_data)

In [ ]:
%%R

first_date_data <- symptomatic_episodes_data %>%
  mutate(
    first_date = map2(symptomatic_episode_date, initial_presentation, ~{
      dates <- .x
      procs <- .y

      if (is.na(procs[1]) || procs[1] != "Colic") {
        return(tibble(first_date = dates[1], initial_presentation = procs[1]))
      }

      within_6m <- which(
        !is.na(procs) &
        procs != "Colic" &
        dates > dates[1] &
        dates <= dates[1] %m+% months(6)
      )

      if (length(within_6m) > 0) {
        idx <- within_6m[1]
        return(tibble(first_date = dates[idx], initial_presentation = procs[idx]))
      }

      tibble(first_date = dates[1], initial_presentation = "Colic")
    })
  ) %>%
  select(-symptomatic_episode_date, -initial_presentation) %>%
  unnest(first_date)

str(first_date_data)

In [ ]:
%%R
table(first_date_data$initial_presentation)

In [ ]:
%%R
table(is.na(combined_data$initial_presentation))

In [ ]:
%%R
n_distinct(first_date_data$person_id)

## Remove Participants with only Kidney Stones Coded

In [ ]:
%%R

table(combined_data2$initial_presentation)

In [ ]:
%%R

combined_data_no_ks <- combined_data2 
str(combined_data_no_ks)

# Build Recurrence Data Structures

This replicates the episode logic from `recurrence_identification_colic_to_colic_multiple_occurrences.R`, adapted for AoU column names.

## Prepare Combined Data

In [ ]:
%%R

# Rename person_id to participant_id for compatibility with downstream analysis code
# Filter out Kidney_stone codes, ensure date format, sort
query_combined_df_clean <- combined_data_no_ks %>%
  rename(participant_id = person_id) %>%
  mutate(date_time = as.Date(date_time)) %>%
  arrange(participant_id, date_time)

str(query_combined_df_clean)

In [ ]:
%%R

n_distinct(query_combined_df_clean$participant_id)

## Pass final_date_df to R

In [ ]:
%%R -i final_date_df

final_date_r <- final_date_df %>%
  as_tibble() %>%
  rename(participant_id = person_id, last_date = final_date) %>%
  mutate(last_date = as.Date(last_date))

str(final_date_r)

## Build dates_detailed (6 month gap)

In [ ]:
%%R

dates_detailed <- query_combined_df_clean %>%
  filter(!(format(date_time, "%m-%d") == "01-01" &
             as.integer(format(date_time, "%Y")) < 2000)) %>%
  group_by(participant_id) %>%
  mutate(
    gap = as.numeric(date_time - lag(date_time)),
    new_episode = is.na(gap) | gap > 182.5,
    episode_num = cumsum(new_episode)
  ) %>%
  filter(new_episode) %>%
  mutate(
    recurrence_num = episode_num - 1,
    episode_gap = as.numeric(date_time - lag(date_time))
  ) %>%
  summarise(
    index_date       = nth(date_time, 1),
    first_date       = nth(date_time, 2),
    second_date      = nth(date_time, 3),
    third_date       = nth(date_time, 4),
    fourth_date      = nth(date_time, 5),
    fifth_date       = nth(date_time, 6),
    n_recurrences    = max(recurrence_num),
    mean_time_between = mean(episode_gap, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  mutate(
    gap_index_to_first_months  = as.numeric(first_date  - index_date)  / 30.44,
    gap_first_to_second_months = as.numeric(second_date - first_date)  / 30.44,
    gap_second_to_third_months = as.numeric(third_date  - second_date) / 30.44
  )

str(dates_detailed)

## Build first_presentation and n_dates

In [ ]:
%%R

first_presentation <- query_combined_df_clean %>%
  group_by(participant_id) %>%
  arrange(date_time) %>%
  slice_head(n = 1) %>%
  ungroup() %>%
  select(participant_id,
         date_of_first_code = date_time,
         initial_presentation)

n_dates <- query_combined_df_clean %>%
  filter(!(format(date_time, "%m-%d") == "01-01" &
             as.integer(format(date_time, "%Y")) < 2000 & n() > 1)) %>%
  arrange(participant_id, date_time) %>%
  group_by(participant_id) %>%
  summarise(
    time_diff = {
      diffs <- as.numeric(difftime(date_time, first(date_time), units = "days"))
      first(diffs[diffs > 182.5], default = NA_real_)
    }
  ) %>%
  filter(!is.na(time_diff)) %>%
  ungroup()

str(first_presentation)
str(n_dates)

## Assemble time_to_first_recurrence1

In [ ]:
%%R

time_to_first_recurrence1 <- first_presentation %>%
  left_join(n_dates, by = "participant_id") %>%
  left_join(final_date_r, by = "participant_id") %>%
  mutate(
    pheno      = ifelse(!is.na(time_diff), 1, 0),
    time_diff  = ifelse(
      !is.na(time_diff),
      time_diff,
      as.numeric(difftime(last_date, date_of_first_code, units = "days"))
    ),
    first_date  = date_of_first_code,
    follow_up   = as.numeric(last_date - date_of_first_code),
    time_diff_years = time_diff / 365
  )

str(time_to_first_recurrence1)
table(time_to_first_recurrence1$pheno)
table(time_to_first_recurrence1$initial_presentation)

In [ ]:
%%R

str(dates_detailed)

## Build overall_follow_up and overall_data_cut_off_5 equivalents

In [ ]:
%%R

# overall_follow_up equivalent (for compatibility with grid search code)
overall_follow_up <- final_date_r %>%
  rename(last_date = last_date)

In [ ]:
%%R

# overall_data_cut_off_5 equivalent
# This counts recurrences within 5 years of the index date
date_cols <- c("first_date", "second_date", "third_date", "fourth_date", "fifth_date")

overall_data_cut_off_5 <- dates_detailed %>%
  left_join(final_date_r, by = "participant_id") %>%
  mutate(
    five_year_cutoff = index_date + (365.25 * 5),
    count_diff = rowSums(
      across(all_of(date_cols), ~ !is.na(.x) & .x <= five_year_cutoff),
      na.rm = TRUE
    )
  ) %>%
  select(participant_id, count_diff, first_date = index_date, last_date)

str(overall_data_cut_off_5)
table(overall_data_cut_off_5$count_diff)

# Process EAU Risk Data

## Load eau_risk.csv

In [ ]:
!gsutil -m cp {bucket}/data/eau_risk.csv .

In [ ]:
%%R

# Copy eau_risk.csv from the workspace bucket

eau_risk <- read.csv("eau_risk.csv") %>% as_tibble() %>%
  mutate(date_time = as.Date(date_time))

str(eau_risk)
table(eau_risk$standard_concept_name)

## Define cohort reference dates

In [ ]:
%%R

# km_recurrence: person_id + index date (used to restrict comorbidities to pre-index)
km_recurrence <- dates_detailed %>%
  select(person_id = participant_id, first_date = index_date)

str(km_recurrence)

## Build pmhx_cleaned from EAU risk concept names

Each binary flag is 1 if any matching concept name appears for that person before their index date.

In [ ]:
%%R

pmhx_cleaned <- eau_risk %>%
  filter(person_id %in% km_recurrence$person_id) %>%
  left_join(km_recurrence %>% select(person_id, first_date),
            by = "person_id",
            relationship = "many-to-many") %>%
  filter(date_time < first_date) %>%
  group_by(person_id, first_date) %>%
  summarise(standard_concept_names = list(unique(standard_concept_name)), .groups = "drop") %>%
  mutate(
    dm = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Diabetes mellitus", "Diabetes mellitus without complication",
      "Type 1 diabetes mellitus", "Type 1 diabetes mellitus without complication",
      "Type 2 diabetes mellitus", "Type 2 diabetes mellitus without complication",
      "Type 2 diabetes mellitus with ulcer", "Type 1 diabetes mellitus with ulcer",
      "Drug-induced diabetes mellitus", "Steroid-induced diabetes",
      "Gestational diabetes mellitus", "Gestational diabetes mellitus class A-1",
      "Gestational diabetes mellitus class A-2",
      "Secondary diabetes mellitus", "Secondary endocrine diabetes mellitus",
      "Posttransplant diabetes mellitus", "Neonatal diabetes mellitus",
      "Latent autoimmune diabetes mellitus in adult",
      "Maturity-onset diabetes of the young",
      "Maturity onset diabetes of the young, type 1",
      "Maturity onset diabetes of the young, type 2",
      "Maturity-onset diabetes of the young, type 3",
      "Maturity-onset diabetes of the young, type 5",
      "Insulin treated type 2 diabetes mellitus",
      "Insulin dependent diabetes mellitus type 1A",
      "Insulin dependent diabetes mellitus type 1B",
      "Ketosis-prone diabetes mellitus", "Lipoatrophic diabetes",
      "Diabetes mellitus associated with cystic fibrosis",
      "Diabetes mellitus associated with pancreatic disease",
      "Diabetes mellitus due to cystic fibrosis",
      "Diabetes mellitus due to pancreatic injury",
      "Diabetes mellitus in mother complicating pregnancy, childbirth AND/OR puerperium",
      "Pre-existing type 1 diabetes mellitus in pregnancy",
      "Pre-existing type 2 diabetes mellitus in pregnancy",
      "Pre-existing diabetes mellitus in pregnancy",
      "Type II diabetes mellitus in remission"
    )))),

    hyperpth = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Hyperparathyroidism", "Primary hyperparathyroidism",
      "Secondary hyperparathyroidism", "Tertiary hyperparathyroidism",
      "Secondary hyperparathyroidism of nonrenal origin",
      "Hyperparathyroidism due to renal insufficiency",
      "Hyperparathyroidism due to vitamin D deficiency",
      "Hyperparathyroidism due to lithium therapy",
      "Familial hyperparathyroidism",
      "Parathyroidectomy", "Complete parathyroidectomy",
      "Subtotal parathyroidectomy", "Parathyroid adenoma excision",
      "Total parathyroidectomy and parathyroid tissue transplantation",
      "Partial parathyroidectomy and parathyroid tissue transposition"
    )))),

    uti = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Urinary tract infectious disease", "Acute urinary tract infection",
      "Chronic urinary tract infection", "Recurrent urinary tract infection",
      "Lower urinary tract infectious disease", "Upper urinary tract infection",
      "Acute upper urinary tract infection", "Acute lower urinary tract infection",
      "Febrile urinary tract infection", "Postoperative urinary tract infection",
      "Catheter-associated urinary tract infection",
      "Urinary tract infection in pregnancy",
      "Urinary tract infection caused by Escherichia coli",
      "Urinary tract infection caused by Enterococcus",
      "Urinary tract infection caused by Klebsiella",
      "Urinary tract infection caused by Pseudomonas",
      "Acute pyelonephritis", "Acute pyelonephritis without medullary necrosis",
      "Acute pyelonephritis with medullary necrosis",
      "Chronic pyelonephritis", "Chronic obstructive pyelonephritis",
      "Chronic pyelonephritis without medullary necrosis",
      "Chronic pyelonephritis with medullary necrosis",
      "Pyelonephritis", "Pyelonephritis in pregnancy",
      "Pyonephrosis", "Infective cystitis", "Infective urethritis",
      "Acute infective cystitis", "Recurrent infective cystitis",
      "Bacterial urinary infection", "Neonatal urinary tract infection",
      "Emphysematous pyelonephritis", "Emphysematous cystitis",
      "Infections of kidney in pregnancy", "Infections of bladder in pregnancy",
      "Infectious disorder of kidney", "Non-obstructive reflux-associated chronic pyelonephritis"
    )))),

    immunosuppression = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Immunosuppression"
    )))),

    transplant_immunosuppression = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Transplanted kidney present", "Transplanted heart present",
      "Transplanted liver present", "Transplanted lung present",
      "Transplanted heart-lung present", "Transplanted cornea present",
      "Transplant of kidney", "Transplantation of heart",
      "Transplantation of liver", "Transplantation of pancreas",
      "Transplantation of lung", "Transplant present",
      "Transplant nephrectomy", "Donor renal transplantation",
      "Live donor renal transplant", "Cadaveric renal transplant",
      "Homotransplant of pancreas", "Orthotopic liver transplant",
      "Heterotopic liver transplant", "Double lung transplant",
      "Single lung transplant", "Autotransplantation of kidney",
      "Renal transplant rejection", "Cardiac transplant rejection",
      "Lung transplant rejection", "Liver transplant rejection",
      "Bone marrow transplant rejection", "Bone marrow transplant present",
      "Bone marrow transplant failure", "Disorder of transplanted kidney",
      "Disorder of transplanted heart", "Disorder of transplanted bone marrow",
      "Complication of transplanted liver", "Complication of transplanted lung",
      "Complication of transplanted pancreas", "Complication of transplanted intestines",
      "Failed renal transplant", "Cardiac transplant failure",
      "Liver transplant failure", "Liver transplant disorder",
      "Heart-lung transplant failure and rejection",
      "Transplanted organ failure", "Transplanted organ rejection",
      "Chronic rejection of renal transplant", "Acute rejection of renal transplant",
      "Accelerated rejection of renal transplant",
      "Delayed renal graft function", "BK virus nephropathy",
      "Transplant glomerulopathy", "Transplant renal artery stenosis",
      "Recurrent post-transplant renal disease",
      "Arteriosclerosis of coronary artery bypass graft of transplanted heart",
      "Coronary arteriosclerosis in artery of transplanted heart",
      "Hypertension secondary to kidney transplant",
      "Hypertension associated with transplantation"
    )))),

    htn = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Essential hypertension", "Benign essential hypertension",
      "Benign hypertension", "Malignant essential hypertension",
      "Malignant hypertension", "Secondary hypertension",
      "Benign secondary hypertension", "Malignant secondary hypertension",
      "Renovascular hypertension", "Renal hypertension",
      "Benign secondary renovascular hypertension",
      "Malignant secondary renovascular hypertension",
      "Hypertensive disorder", "Hypertensive crisis",
      "Hypertensive emergency", "Hypertensive urgency",
      "Pregnancy-induced hypertension", "Maternal hypertension",
      "Hypertension secondary to endocrine disorder",
      "Hypertension complicating pregnancy, childbirth and the puerperium",
      "Pre-existing hypertension complicating pregnancy, childbirth and puerperium",
      "Labile essential hypertension", "Labile systemic arterial hypertension",
      "Resistant hypertensive disorder", "Paroxysmal hypertension",
      "Diastolic hypertension", "Systolic hypertension",
      "Systolic essential hypertension", "Postoperative hypertension",
      "Transient hypertension", "Intermittent hypertension"
    )))),

    dyslipidaemia = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Dyslipidemia", "Complex dyslipidemia",
      "Dyslipidemia due to type 1 diabetes mellitus",
      "Dyslipidemia due to type 2 diabetes mellitus",
      "Dyslipidemia with high density lipoprotein below reference range and triglyceride above reference range due to type 2 diabetes mellitus",
      "Metabolic syndrome X"
    )))),

    pkd = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Autosomal dominant polycystic kidney disease",
      "Autosomal dominant polycystic kidney disease in childhood",
      "Polycystic kidney disease, infantile type",
      "Acquired polycystic kidney disease",
      "Multiple congenital cysts of kidney", "Multiple renal cysts",
      "Medullary sponge kidney", "Bilateral medullary sponge kidney",
      "Multicystic renal dysplasia", "Infected renal cyst"
    )))),

    msk = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Lumbar sprain", "Cervical spine sprain", "Thoracic back sprain",
      "Sprain of spinal ligament", "Sprain of ligament of lumbosacral joint",
      "Sprain of sacrococcygeal ligament",
      "Annular tear of lumbar disc", "Intervertebral disc rupture",
      "Rupture of lumbar intervertebral disc",
      "Tear of the annulus fibrosus of intervertebral disc",
      "Fatigue fracture of vertebra", "Osteoporotic fracture of vertebra",
      "Osteoporotic vertebral collapse", "Pathological fracture of vertebra"
    )))),

    bowel_surgery = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Total colectomy", "Laparoscopic total colectomy",
      "Right colectomy", "Left colectomy", "Extended right hemicolectomy",
      "Sigmoid colectomy", "Ileocolic resection", "Rectosigmoidectomy",
      "Abdominoperineal resection", "Anterior resection of rectum with colostomy",
      "Low anterior resection of rectum", "Hartmann operation, rectal resection",
      "Partial colectomy with anastomosis", "Partial colectomy with coloproctostomy",
      "Partial colectomy with coloproctostomy and colostomy",
      "Partial resection of colon", "Partial excision of large intestine",
      "Large intestine excision", "Excision of colon", "Excision of colon and rectum",
      "Laparoscopic excision of part of colon",
      "Laparoscopic-assisted right colectomy", "Laparoscopic-assisted sigmoid colectomy",
      "Laparoscopic left hemicolectomy", "Hand assisted laparoscopic left colectomy",
      "Small intestine excision", "Total excision of small intestine",
      "Laparoscopic excision of small intestine",
      "Multiple segmental resections of small intestine",
      "Resection of exteriorized segment of small intestine",
      "Pancreaticoduodenectomy", "Radical pancreaticoduodenectomy",
      "Duodenectomy", "Ileectomy and ileostomy", "Incidental appendectomy",
      "Total abdominal colectomy with proctectomy and ileostomy",
      "Total abdominal colectomy with proctectomy and continent ileostomy",
      "Proctopexy combined with sigmoid resection by abdominal approach",
      "Excision of rectal procidentia with anastomosis by perineal approach",
      "Excision of lesion of large intestine", "Excision of lesion of small intestine",
      "Total resection of large intestine"
    )))),

    ibd = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Crohn's disease", "Crohn's disease of small intestine",
      "Crohn's disease of large bowel", "Crohn's disease of colon",
      "Crohn's disease of small AND large intestines",
      "Crohn's disease of terminal ileum", "Crohn's disease of ileum",
      "Crohn's disease of rectum", "Crohn's disease of duodenum",
      "Crohn's disease of intestine", "Crohn's disease in remission",
      "Gastrointestinal Crohn's disease", "Crohn disease of anal canal",
      "Ulcerative colitis", "Chronic ulcerative colitis",
      "Acute ulcerative colitis", "Chronic ulcerative pancolitis",
      "Chronic ulcerative enterocolitis", "Chronic ulcerative ileocolitis",
      "Chronic ulcerative rectosigmoiditis", "Chronic left-sided ulcerative colitis",
      "Left sided ulcerative colitis", "Ulcerative pancolitis",
      "Ulcerative proctocolitis", "Indeterminate colitis",
      "Microscopic colitis", "Collagenous colitis",
      "Inflammatory bowel disease", "Noninfectious colitis"
    )))),

    malabsorption = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Intestinal malabsorption", "Malabsorption syndrome",
      "Post-surgical malabsorption", "Short bowel syndrome",
      "Blind loop syndrome", "Celiac disease", "Adult form of celiac disease",
      "Sprue", "Tropical sprue", "Whipple's disease",
      "Exocrine pancreatic insufficiency", "Pancreatic insufficiency",
      "Secondary pancreatic insufficiency", "Pancreatic malabsorption",
      "Pancreatic steatorrhea", "Chronic steatorrhea",
      "Intestinal disaccharidase deficiency", "Sucrase-isomaltase deficiency",
      "Malabsorption syndrome due to intolerance to lactose",
      "Nonpersistence of intestinal lactase",
      "Bile acid malabsorption syndrome", "Bile acid malabsorption syndrome type III",
      "Postcholecystectomy diarrhea", "Protein-losing enteropathy",
      "Intestinal malabsorption of fat", "Malabsorption - iron",
      "Post-gastrointestinal tract surgery malnutrition",
      "Exocrine pancreatic manifestation co-occurrent and due to cystic fibrosis"
    )))),

    sarcoidosis = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Sarcoidosis", "Pulmonary sarcoidosis", "Cardiac sarcoidosis",
      "Neurosarcoidosis", "Cutaneous sarcoidosis", "Lymph node sarcoidosis",
      "Sarcoid heart muscle disease", "Sarcoid arthropathy", "Sarcoid arthritis",
      "Sarcoid myopathy", "Sarcoid neuropathy", "Sarcoid meningitis",
      "Subcutaneous sarcoidosis", "Sarcoidosis of digestive system",
      "Sarcoidosis of lung with sarcoidosis of lymph nodes",
      "Sarcoidosis, Darier-Roussy type", "Sarcoidosis, lupus pernio type",
      "Stage 3 pulmonary sarcoidosis", "Heerfordt's syndrome", "Lofgrens syndrome",
      "Demyelination of central nervous system co-occurrent and due to neurosarcoidosis",
      "Granulomatous sarcoid nephropathy"
    )))),

    spinal_injury = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Spinal cord injury", "Spinal cord injury without spinal bone injury",
      "Cervical spinal cord injury", "Thoracic cord injury without spinal bone injury",
      "Injury of cervical spinal cord", "Injury of thoracic spinal cord",
      "Injury of lumbar spinal cord", "Late effect of spinal cord injury",
      "Fracture of cervical spine", "Fracture of lumbar spine",
      "Fracture of thoracic spine", "Fracture of sacrum",
      "Closed fracture of cervical spine", "Closed fracture lumbar vertebra",
      "Closed fracture thoracic vertebra", "Traumatic spondylopathy",
      "Spinal dislocation with cervical cord lesion",
      "Spinal dislocation with thoracic cord lesion",
      "Spinal dislocation with lumbar cord lesion",
      "Complete lesion of cervical spinal cord", "Complete lesion of thoracic spinal cord",
      "Complete lesion of lumbar spinal cord"
    )))),

    cf = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Cystic fibrosis", "Cystic fibrosis of the lung",
      "Cystic fibrosis with meconium ileus", "Cystic fibrosis without meconium ileus",
      "Diabetes mellitus associated with cystic fibrosis",
      "Diabetes mellitus due to cystic fibrosis",
      "Pancreatic insufficiency due to cystic fibrosis of pancreas",
      "Exocrine pancreatic manifestation co-occurrent and due to cystic fibrosis"
    )))),

    pujo = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Congenital obstruction of ureteropelvic junction",
      "Congenital hydronephrosis due to ureteropelvic junction obstruction",
      "Acquired hydronephrosis due to ureteropelvic junction obstruction",
      "Obstruction of pelviureteric junction"
    )))),

    diverticulum = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Calcyceal diverticulum"
    )))),

    ureteric_stricture = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Stricture of ureter", "Acquired stricture of ureter",
      "Postoperative ureteric constriction"
    )))),

    vur = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Vesicoureteric reflux", "Vesicoureteral reflux without reflux nephropathy",
      "Congenital vesicoureterorenal reflux",
      "Secondary vesicoureteric reflux",
      "Secondary right vesicoureteral reflux grade 3",
      "Non-obstructive reflux-associated chronic pyelonephritis"
    )))),

    single_kidney = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Total nephrectomy", "Radical nephrectomy", "Recipient nephrectomy",
      "Transplant nephrectomy"
    )))),

    ileal_conduit = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Endoscopy of ileal conduit", "Ureteroileostomy"
    )))),

    eating_disorder = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Eating disorder", "Eating disorder in remission",
      "Anorexia nervosa", "Anorexia nervosa in remission",
      "Anorexia nervosa, restricting type", "Anorexia nervosa, binge-eating purging type",
      "Atypical anorexia nervosa", "Bulimia nervosa",
      "Bulimia nervosa in partial remission", "Bulimia nervosa, purging type",
      "Avoidant restrictive food intake disorder",
      "Rumination disorder", "Rumination disorder of infancy",
      "Adult rumination syndrome of ingested food",
      "Pica", "Pica of infancy and childhood", "Psychogenic overeating"
    )))),

    horseshoe_kidney = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Horseshoe kidney"
    )))),

    .keep = "all"
  ) %>%
    mutate(

  .impaired_glucose = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
    "Diabetes mellitus without complication",
    "Diabetes mellitus",
    "Type 2 diabetes mellitus without complication",
    "Type 2 diabetes mellitus",
    "Gestational diabetes mellitus"

  )))),

  .central_obesity = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
    "Metabolic syndrome X",
      "High BMI",
      "Obesity"

  )))),

  .dyslipidaemia_metabolic = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
    "Dyslipidemia",
    "Complex dyslipidemia",
    "Dyslipidemia due to type 2 diabetes mellitus",
    "Dyslipidemia due to type 1 diabetes mellitus",
    "Dyslipidemia with high density lipoprotein below reference range and triglyceride above reference range due to type 2 diabetes mellitus"
  )))),

  .htn_metabolic = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
    "Essential hypertension", "Benign essential hypertension",
    "Hypertensive disorder", "Secondary hypertension"
  )))),

  # Metabolic disorder = 3 or more of the 4 components
  metabolic_disorder = as.integer(
    (.impaired_glucose + .central_obesity + .dyslipidaemia_metabolic + .htn_metabolic) >= 3
  ),
        .keep = "all"
) %>%
select(-starts_with("."))  %>%
select(-standard_concept_names)

str(pmhx_cleaned)

## Replace UTI with Recurrent UTI Flag

In [ ]:
%%R

recurrent_uti_ids <- eau_risk %>%
  filter(person_id %in% km_recurrence$person_id) %>%
  left_join(
    km_recurrence %>% select(person_id, first_date),
    by = "person_id",
    relationship = "many-to-many"
  ) %>%
  filter(date_time < first_date) %>%
  filter(standard_concept_name %in% c(
    "Urinary tract infectious disease", "Acute urinary tract infection",
    "Chronic urinary tract infection", "Recurrent urinary tract infection",
    "Lower urinary tract infectious disease", "Upper urinary tract infection",
    "Acute upper urinary tract infection", "Acute lower urinary tract infection",
    "Febrile urinary tract infection", "Postoperative urinary tract infection",
    "Catheter-associated urinary tract infection",
    "Urinary tract infection in pregnancy",
    "Urinary tract infection caused by Escherichia coli",
    "Urinary tract infection caused by Enterococcus",
    "Urinary tract infection caused by Klebsiella",
    "Urinary tract infection caused by Pseudomonas",
    "Acute pyelonephritis", "Acute pyelonephritis without medullary necrosis",
    "Acute pyelonephritis with medullary necrosis",
    "Chronic pyelonephritis", "Chronic obstructive pyelonephritis",
    "Chronic pyelonephritis without medullary necrosis",
    "Chronic pyelonephritis with medullary necrosis",
    "Pyelonephritis", "Pyelonephritis in pregnancy",
    "Pyonephrosis", "Infective cystitis", "Infective urethritis",
    "Acute infective cystitis", "Recurrent infective cystitis",
    "Bacterial urinary infection", "Neonatal urinary tract infection",
    "Emphysematous pyelonephritis", "Emphysematous cystitis",
    "Infections of kidney in pregnancy", "Infections of bladder in pregnancy",
    "Infectious disorder of kidney",
    "Non-obstructive reflux-associated chronic pyelonephritis"
  )) %>%
  distinct(person_id, date_time) %>%
  arrange(person_id, date_time) %>%
  mutate(date_time = anydate(date_time)) %>%
  group_by(person_id) %>%
  mutate(
    within_6mo = as.integer(
      as.numeric(lead(date_time) - date_time) <= 180
    ),
    within_1yr = as.integer(
      as.numeric(lead(date_time, 2) - date_time) <= 365
    )
  ) %>%
  summarise(
    recurrent_uti = as.integer(
      any(within_6mo == 1, na.rm = TRUE) |
      any(within_1yr == 1, na.rm = TRUE)
    )
  ) %>%
  ungroup()

# Replace the uti flag with recurrent UTI
pmhx_cleaned <- pmhx_cleaned %>%
  left_join(recurrent_uti_ids, by = "person_id") %>%
  mutate(
    uti = as.integer(coalesce(recurrent_uti, 0L))
  ) %>%
  select(-recurrent_uti)

table(pmhx_cleaned$uti)

## Define EAU Risk Factor Columns

In [ ]:
%%R

eau_risk_cols <- c("metabolic_disorder", "hyperpth",
                   "immunosuppression", "transplant_immunosuppression",
                   "pkd", "msk", "bowel_surgery", "ibd", "malabsorption",
                   "sarcoidosis", "spinal_injury", "cf", "pujo", "diverticulum",
                   "ureteric_stricture", "vur", "single_kidney",
                   "ileal_conduit", "eating_disorder", "horseshoe_kidney")

# Compute Age at First Stone

In [ ]:
%%R -i overall_person_df

dob_df <- overall_person_df %>%
  as_tibble() %>%
  rename(participant_id = person_id) %>%
  mutate(dob = as.Date(date_of_birth)) %>%
  select(participant_id, dob)

age_at_first_stone_df <- dates_detailed %>%
  select(participant_id, index_date) %>%
  left_join(dob_df, by = "participant_id") %>%
  mutate(age_at_first_stone = as.numeric((index_date - dob) / 365.25)) %>%
  select(participant_id, age_at_first_stone)

str(age_at_first_stone_df)
summary(age_at_first_stone_df$age_at_first_stone)

# EAU High-Risk Classification Function

Builds high-risk classification for a given starting point.
Age < 25 is evaluated at the relevant starting date.
Two versions: with and without UTI.

In [ ]:
%%R

build_high_risk_df <- function(starting_point_col, include_uti = FALSE) {
  ref_dates <- dates_detailed %>%
    dplyr::select(participant_id, ref_date = !!sym(starting_point_col)) %>%
    filter(!is.na(ref_date)) %>%
    distinct(participant_id, .keep_all = TRUE)

  age_at_start <- ref_dates %>%
    left_join(dob_df, by = "participant_id") %>%
    mutate(age_at_start = as.numeric((ref_date - dob) / 365.25)) %>%
    dplyr::select(participant_id, age_at_start)

  risk_cols <- if (include_uti) c(eau_risk_cols, "uti") else eau_risk_cols

  pmhx_for_join <- pmhx_cleaned %>% rename(participant_id = person_id)

  dates_detailed %>%
    dplyr::select(participant_id) %>%
    distinct() %>%
    left_join(pmhx_for_join, by = "participant_id") %>%
    mutate(across(any_of(risk_cols), ~replace_na(., 0))) %>%
    mutate(
      risk_sum  = rowSums(across(any_of(risk_cols)), na.rm = TRUE),
      high_risk = if_else(risk_sum < 1, "low_risk", "high_risk")
    ) %>%
    dplyr::select(participant_id, high_risk) %>%
    left_join(age_at_start, by = "participant_id") %>%
    mutate(high_risk = case_when(age_at_start < 25 ~ "high_risk", TRUE ~ high_risk)) %>%
    group_by(participant_id) %>%
    summarise(high_risk = if_else(any(high_risk == "high_risk"), "high_risk", "low_risk"),
              .groups = "drop")
}

# Preview: index_date versions
cat("Without UTI (index_date):\n")
table(build_high_risk_df("index_date", include_uti = FALSE)$high_risk)
cat("\nWith UTI (index_date):\n")
table(build_high_risk_df("index_date", include_uti = TRUE)$high_risk)

# With vs Without UTI Analyses

Replicates the analyses from `with_vs_without_uti_for_gh.Rmd`.

## Time-to-Recurrence Dataset

In [ ]:
%%R

time_to_recurrence_2 <- time_to_first_recurrence1 %>%
  mutate(
    time_0 = first_date,
    recurrence = ifelse(pheno > 0, "rsf", "ssf"),
    time_to_recurrence_1 = time_diff_years
  )

str(time_to_recurrence_2)

## Overall Kaplan-Meier

In [ ]:
%%R

demographics_km1 <- time_to_recurrence_2 %>%
  subset(select = c(recurrence, time_to_recurrence_1)) %>%
  drop_na(recurrence) %>%
  drop_na(time_to_recurrence_1)
demographics_km1$recurrence <- ifelse(demographics_km1$recurrence == "rsf", 1, 0)
demographics_km1$recurrence <- as.integer(demographics_km1$recurrence)

km_fit <- surv_fit(Surv(time_to_recurrence_1, as.integer(recurrence)) ~ 1,
                   data = demographics_km1)

ggsurvplot(km_fit,
           data = demographics_km1,
           risk.table = TRUE,
           surv.median.line = "hv",
           conf.int = TRUE,
           tables.theme = theme_cleantable(),
           break.time.by = 1,
           xlab = "Time (years)",
           xlim = c(0, 10),
           title = "Time to First Recurrence",
           censor = FALSE)

In [ ]:
%%R

as_tibble(cbind(
  "Yearly Recurrence" = c(2, 5, 10),
  "Recurrence Rates" = 1 - round(summary(km_fit, times = c(2, 5, 10))$surv, digits = 2),
  "95% CI" = paste0(
    1 - round(summary(km_fit, times = c(2, 5, 10))$upper, digits = 2),
    "-",
    1 - round(summary(km_fit, times = c(2, 5, 10))$lower, digits = 2)
  )
)) 

## Age at First Stone (for with vs without UTI)

In [ ]:
%%R

age_at_first_stone_df_2 <- time_to_recurrence_2 %>%
  subset(select = c(participant_id, recurrence, time_to_recurrence_1)) %>%
  left_join(pmhx_cleaned %>% rename(participant_id = person_id), by = "participant_id") %>%
  left_join(dob_df, by = "participant_id") %>%
  left_join(time_to_first_recurrence1 %>% dplyr::select(participant_id, date_of_first_code),
            by = "participant_id") %>%
  mutate(age_at_first_stone = (date_of_first_code - dob) / 365.25) %>%
  dplyr::select(participant_id, age_at_first_stone)

str(age_at_first_stone_df_2)

## Risk Stratification Without UTI

In [ ]:
%%R

high_risk_without_uti <- time_to_first_recurrence1 %>%
  select(participant_id) %>%
  distinct(participant_id) %>%
  left_join(pmhx_cleaned %>% rename(participant_id = person_id), by = "participant_id") %>%
  dplyr::select(-uti) %>%
  mutate(across(any_of(c(eau_risk_cols)), ~replace_na(., 0))) %>%
  mutate(
    risk_sum = rowSums(across(any_of(eau_risk_cols)), na.rm = TRUE),
    high_risk = if_else(risk_sum < 1, "low_risk", "high_risk")
  ) %>%
  select(participant_id, high_risk) %>%
  left_join(age_at_first_stone_df_2, by = "participant_id") %>%
  mutate(high_risk = case_when(age_at_first_stone < 25 ~ "high_risk", TRUE ~ high_risk)) %>%
  group_by(participant_id) %>%
  summarise(high_risk = if_else(any(high_risk == "high_risk"), "high_risk", "low_risk"),
            .groups = "drop")

table(high_risk_without_uti$high_risk)

## Risk Stratification With UTI

In [ ]:
%%R

high_risk_with_uti <- time_to_first_recurrence1 %>%
  select(participant_id) %>%
  distinct(participant_id) %>%
  left_join(pmhx_cleaned %>% rename(participant_id = person_id), by = "participant_id") %>%
  mutate(across(any_of(c(eau_risk_cols, "uti")), ~replace_na(., 0))) %>%
  mutate(
    risk_sum = rowSums(across(any_of(c(eau_risk_cols, "uti"))), na.rm = TRUE),
    high_risk = if_else(risk_sum < 1, "low_risk", "high_risk")
  ) %>%
  select(participant_id, high_risk) %>%
  left_join(age_at_first_stone_df_2, by = "participant_id") %>%
  mutate(high_risk = case_when(age_at_first_stone < 25 ~ "high_risk", TRUE ~ high_risk)) %>%
  group_by(participant_id) %>%
  summarise(high_risk = if_else(any(high_risk == "high_risk"), "high_risk", "low_risk"),
            .groups = "drop")

table(high_risk_with_uti$high_risk)

## High-Risk Classification (with initial presentation)

In [ ]:
%%R

high_risk <- time_to_recurrence_2 %>%
  subset(select = c(participant_id, recurrence, time_to_recurrence_1, initial_presentation)) %>%
  left_join(pmhx_cleaned %>% rename(participant_id = person_id), by = "participant_id") %>%
  left_join(dob_df, by = "participant_id") %>%
  left_join(time_to_first_recurrence1 %>% dplyr::select(participant_id, date_of_first_code),
            by = "participant_id") %>%
  mutate(
    age_at_first_stone = (date_of_first_code - dob) / 365.25,
    risk_sum = rowSums(across(any_of(eau_risk_cols)), na.rm = TRUE),
    high_risk = case_when(
      age_at_first_stone < 25 ~ "high_risk",
      risk_sum >= 1 ~ "high_risk",
      TRUE ~ "low_risk"
    )
  ) %>%
  dplyr::select(participant_id, initial_presentation, high_risk)

table(high_risk$high_risk)

## Assemble Base Datasets

In [ ]:
%%R

### Unmatched: Without UTI
base_data_without_uti <- overall_data_cut_off_5 %>%
  left_join(high_risk_without_uti, by = "participant_id") %>%
  left_join(high_risk %>% dplyr::select(participant_id, initial_presentation), by = "participant_id") %>%
  left_join(age_at_first_stone_df_2, by = "participant_id") %>%
  dplyr::select(participant_id, count_diff, first_date, last_date, high_risk, initial_presentation, age_at_first_stone) %>%
  mutate(recurrence = if_else(count_diff == 0, 0, 1), .keep = "all")

### Unmatched: With UTI
base_data_with_uti <- overall_data_cut_off_5 %>%
  left_join(high_risk_with_uti, by = "participant_id") %>%
  left_join(high_risk %>% dplyr::select(participant_id, initial_presentation), by = "participant_id") %>%
  left_join(age_at_first_stone_df_2, by = "participant_id") %>%
  dplyr::select(participant_id, count_diff, first_date, last_date, high_risk, initial_presentation, age_at_first_stone) %>%
  mutate(recurrence = if_else(count_diff == 0, 0, 1), .keep = "all")

### KM datasets
base_data_without_uti_for_km <- time_to_first_recurrence1 %>%
  left_join(high_risk_without_uti, by = "participant_id") %>%
  left_join(age_at_first_stone_df_2, by = "participant_id") %>%
  dplyr::select(participant_id, pheno, high_risk, initial_presentation, age_at_first_stone, time_diff_years) %>%
  mutate(recurrence = pheno,
         age_at_first_stone = as.numeric(age_at_first_stone),
         time_to_recurrence_1 = time_diff_years,
         .keep = "unused") %>%
  drop_na(time_to_recurrence_1)

base_data_with_uti_for_km <- time_to_first_recurrence1 %>%
  left_join(high_risk_with_uti, by = "participant_id") %>%
  left_join(age_at_first_stone_df_2, by = "participant_id") %>%
  dplyr::select(participant_id, pheno, high_risk, initial_presentation, age_at_first_stone, time_diff_years) %>%
  mutate(recurrence = pheno,
         age_at_first_stone = as.numeric(age_at_first_stone),
         time_to_recurrence_1 = time_diff_years,
         .keep = "unused") %>%
  drop_na(time_to_recurrence_1)

str(base_data_without_uti)
str(base_data_with_uti)

## Summary Tables

### Without UTI

In [ ]:
%%R

base_data_without_uti_for_km_summary <- base_data_without_uti_for_km %>%
  left_join(pmhx_cleaned %>% rename(participant_id = person_id), by = "participant_id") %>%
  dplyr::select(-c(time_to_recurrence_1, participant_id)) %>%  
  mutate(early_onset = case_when(age_at_first_stone < 25 ~ "Yes", TRUE ~ "No"),
         across(where(is.integer), ~ replace_na(.x, 0)),
         .keep = "all")

In [ ]:
%%R

# Identify column types
chr_vars <- base_data_without_uti_for_km_summary %>%
  select(where(is.character)) %>% names()

int_vars <- base_data_without_uti_for_km_summary %>%
  select(where(~ is.integer(.x) | is.numeric(.x)), -age_at_first_stone) %>% names()

# Categorical: character variables
chr_summary <- base_data_without_uti_for_km_summary %>%
  select(all_of(chr_vars)) %>%
  pivot_longer(everything(), names_to = "variable", values_to = "value") %>%
  count(variable, value) %>%
  group_by(variable) %>%
  mutate(pct = round(n / sum(n) * 100, 1),
         summary = paste0(n, " (", pct, "%)")) %>%
  ungroup()

# Categorical: binary integer variables (0/1)
int_summary <- base_data_without_uti_for_km_summary %>%
  select(all_of(int_vars)) %>%
  pivot_longer(everything(), names_to = "variable", values_to = "value") %>%
  count(variable, value) %>%
  group_by(variable) %>%
  mutate(pct = round(n / sum(n) * 100, 1),
         summary = paste0(n, " (", pct, "%)")) %>%
  ungroup()

categorical_summary <- bind_rows(chr_summary, int_summary %>% mutate(value = as.character(value)))

In [ ]:
%%R

chr_summary %>%
    filter(value != "No") %>%
    filter(value != "0") %>%
    filter(value != "low_risk") %>%
    dplyr::select(variable, value, summary) 

In [ ]:
%%R

categorical_summary %>%
    filter(value != "No") %>%
    filter(value != "0") %>%
    filter(value != "low_risk") %>%
    arrange(desc(pct)) %>%
    dplyr::select(variable, summary) %>%
    print(n=30)

### With UTI

In [ ]:
%%R

base_data_with_uti_for_km_summary <- base_data_with_uti_for_km %>%
  dplyr::select(high_risk) %>%
    group_by(high_risk) %>%
    summarise(
        count = n()
    )

base_data_with_uti_for_km_summary

## Analysis Parameters and Matched Datasets

In [ ]:
%%R

thresholds <- c(1, 2, 3)
labels <- c("0 vs ≥1 events", "0 vs ≥2 events", "0 vs ≥3 events")
colors <- c("#1b9e77", "#d95f02", "#7570b3")

### Propensity-Score Matched Datasets

In [ ]:
%%R

### Matched: Without UTI
base_data_match_without_uti <- base_data_without_uti %>%
  mutate(
    high_risk_binary = if_else(high_risk == "high_risk", 1L, 0L),
    age_at_first_stone = as.numeric(age_at_first_stone),
    initial_presentation = as.factor(initial_presentation)
  ) %>%
  drop_na(high_risk_binary, age_at_first_stone, initial_presentation)

m <- matchit(high_risk_binary ~ age_at_first_stone + initial_presentation,
             data = base_data_match_without_uti, method = "nearest")
base_data_match_without_uti <- match.data(m)

### Matched: With UTI
base_data_match_with_uti <- base_data_with_uti %>%
  mutate(
    high_risk_binary = if_else(high_risk == "high_risk", 1L, 0L),
    age_at_first_stone = as.numeric(age_at_first_stone),
    initial_presentation = as.factor(initial_presentation)
  ) %>%
  drop_na(high_risk_binary, age_at_first_stone, initial_presentation)

m <- matchit(high_risk_binary ~ age_at_first_stone + initial_presentation,
             data = base_data_match_with_uti, method = "nearest")
base_data_match_with_uti <- match.data(m)

## Helper Functions

In [ ]:
%%R

prep_roc <- function(base_data, thresh) {
  base_data %>%
    mutate(
      actual = case_when(
        count_diff >= thresh ~ 1,
        count_diff == 0 ~ 0,
        TRUE ~ NA_integer_
      ),
      predicted = if_else(high_risk == "high_risk", 1, 0)
    ) %>%
    drop_na()
}

extract_metrics <- function(roc_obj, dataset_name, label, p_text) {
  auc_val <- as.numeric(auc(roc_obj))
  auc_ci  <- ci.auc(roc_obj)
  coords_best <- coords(roc_obj, "best", ret = c("sensitivity", "specificity"),
                        best.method = "youden")
  sens_ci <- ci.se(roc_obj, specificities = coords_best$specificity)
  spec_ci <- ci.sp(roc_obj, sensitivities = coords_best$sensitivity)

  data.frame(
    Dataset = dataset_name,
    Comparison = label,
    AUC_CI = sprintf("%.2f (%.2f–%.2f)", auc_val, auc_ci[1], auc_ci[3]),
    Sensitivity = sprintf("%.2f (%.2f–%.2f)",
                          coords_best$sensitivity, sens_ci[, 1], sens_ci[, 3]),
    Specificity = sprintf("%.2f (%.2f–%.2f)",
                          coords_best$specificity, spec_ci[, 1], spec_ci[, 3]),
    DeLong_p = p_text
  )
}

generate_roc_plot <- function(base_data, plot_title) {
  roc_list <- list()
  auc_table_df <- data.frame(Comparison = character(), AUC_CI = character())
  ci_combined_df <- data.frame()

  for (i in seq_along(thresholds)) {
    thresh <- thresholds[i]
    label <- labels[i]
    roc_data <- prep_roc(base_data, thresh)
    roc_obj <- roc(roc_data$actual, roc_data$predicted, quiet = TRUE)
    roc_list[[i]] <- list(roc_obj = roc_obj, color = colors[i], label = label)
    auc_val <- as.numeric(auc(roc_obj))
    auc_ci  <- ci.auc(roc_obj)
    auc_table_df <- rbind(auc_table_df,
      data.frame(Comparison = label,
                 AUC_CI = sprintf("%.2f (%.2f–%.2f)", auc_val, auc_ci[1], auc_ci[3])))
    ci_se <- ci.se(roc_obj, specificities = seq(0, 1, length.out = 100))
    ci_combined_df <- rbind(ci_combined_df, data.frame(
      fpr = 1 - as.numeric(rownames(ci_se)),
      tpr_lower = ci_se[, 1], tpr_upper = ci_se[, 3], Comparison = label))
  }

  roc_combined_df <- do.call(rbind, lapply(seq_along(roc_list), function(i) {
    roc_obj <- roc_list[[i]]$roc_obj
    data.frame(fpr = 1 - roc_obj$specificities,
               tpr = roc_obj$sensitivities,
               Comparison = roc_list[[i]]$label)
  }))

  auc_table <- tableGrob(auc_table_df, rows = NULL, theme = ttheme_minimal(base_size = 9))
  auc_table_boxed <- grobTree(
    rectGrob(gp = gpar(col = "black", fill = "white", lwd = 1)), auc_table)

  ggplot() +
    geom_ribbon(data = ci_combined_df,
                aes(x = fpr, ymin = tpr_lower, ymax = tpr_upper, fill = Comparison), alpha = 0.2) +
    geom_line(data = roc_combined_df,
              aes(x = fpr, y = tpr, color = Comparison), linewidth = 1) +
    geom_abline(linetype = "dashed", alpha = 0.6) +
    labs(x = "False Positive Rate", y = "True Positive Rate", title = plot_title) +
    scale_color_manual(values = colors) + scale_fill_manual(values = colors) +
    coord_equal() + theme_minimal(base_size = 12) +
    annotation_custom(grob = auc_table_boxed,
                      xmin = 0.55, xmax = 1, ymin = 0.05, ymax = 0.3) + theme_bw()
}

generate_faceted_roc_plot <- function(base_data, plot_title) {
  presentations <- unique(base_data$initial_presentation)
  roc_combined_df <- data.frame()
  ci_combined_df  <- data.frame()
  auc_table_df    <- data.frame()

  for (pres in presentations) {
    pres_data <- base_data %>% filter(initial_presentation == pres)
    for (i in seq_along(thresholds)) {
      thresh <- thresholds[i]
      label  <- labels[i]
      roc_data <- pres_data %>%
        mutate(actual = case_when(count_diff >= thresh ~ 1, count_diff == 0 ~ 0, TRUE ~ NA_integer_),
               predicted = if_else(high_risk == "high_risk", 1, 0)) %>%
        drop_na()
      if (nrow(roc_data) < 10 || length(unique(roc_data$actual)) < 2) next
      roc_obj <- roc(roc_data$actual, roc_data$predicted, quiet = TRUE)
      roc_combined_df <- rbind(roc_combined_df, data.frame(
        fpr = 1 - roc_obj$specificities, tpr = roc_obj$sensitivities,
        Comparison = label, initial_presentation = pres))
      ci_se <- ci.se(roc_obj, specificities = seq(0, 1, length.out = 100))
      ci_combined_df <- rbind(ci_combined_df, data.frame(
        fpr = 1 - as.numeric(rownames(ci_se)),
        tpr_lower = ci_se[, 1], tpr_upper = ci_se[, 3],
        Comparison = label, initial_presentation = pres))
      auc_val <- as.numeric(auc(roc_obj))
      auc_ci  <- ci.auc(roc_obj)
      auc_table_df <- rbind(auc_table_df, data.frame(
        initial_presentation = pres, Comparison = label,
        AUC = sprintf("%.2f (%.2f–%.2f)", auc_val, auc_ci[1], auc_ci[3])))
    }
  }

  ggplot() +
    geom_ribbon(data = ci_combined_df,
                aes(x = fpr, ymin = tpr_lower, ymax = tpr_upper, fill = Comparison), alpha = 0.2) +
    geom_line(data = roc_combined_df,
              aes(x = fpr, y = tpr, color = Comparison), linewidth = 1) +
    geom_abline(linetype = "dashed", alpha = 0.6) +
    geom_label(data = auc_table_df,
      aes(label = paste0(Comparison, ": ", AUC), color = Comparison),
      x = 0.95, y = seq(0.25, 0.05, length.out = length(labels))[
        match(auc_table_df$Comparison, labels)],
      hjust = 1, size = 2.8, fill = "white", label.size = 0.3, show.legend = FALSE) +
    scale_color_manual(values = setNames(colors, labels)) +
    scale_fill_manual(values = setNames(colors, labels)) +
    facet_wrap(~ initial_presentation) + coord_equal() +
    labs(x = "False Positive Rate", y = "True Positive Rate",
         title = plot_title, color = NULL, fill = NULL) +
    theme_bw(base_size = 11) +
    theme(legend.position = "bottom", strip.text = element_text(face = "bold"))
}

## DeLong's Tests

### Unmatched: Without UTI vs With UTI

In [ ]:
%%R

results_unmatched_df <- data.frame(
  Dataset = character(), Comparison = character(),
  AUC_CI = character(), Sensitivity = character(),
  Specificity = character(), DeLong_p = character(),
  stringsAsFactors = FALSE
)

for (i in seq_along(thresholds)) {
  thresh <- thresholds[i]
  roc_with    <- prep_roc(base_data_with_uti, thresh)
  roc_without <- prep_roc(base_data_without_uti, thresh)
  roc_obj1 <- roc(roc_with$actual, roc_with$predicted, quiet = TRUE)
  roc_obj2 <- roc(roc_without$actual, roc_without$predicted, quiet = TRUE)
  test_result <- roc.test(roc_obj1, roc_obj2, method = "delong")
  p_val <- test_result$p.value
  p_label <- if_else(p_val < 0.001, "<0.001", sprintf("%.3f", p_val))
  results_unmatched_df <- rbind(results_unmatched_df,
    extract_metrics(roc_obj2, "Without UTI", labels[i], p_label),
    extract_metrics(roc_obj1, "With UTI", labels[i], ""))
}

results_unmatched_df 

### Matched: Without UTI vs With UTI

In [ ]:
%%R

results_matched_df <- data.frame(
  Dataset = character(), Comparison = character(),
  AUC_CI = character(), Sensitivity = character(),
  Specificity = character(), DeLong_p = character(),
  stringsAsFactors = FALSE
)

for (i in seq_along(thresholds)) {
  thresh <- thresholds[i]
  roc_with    <- prep_roc(base_data_match_with_uti, thresh)
  roc_without <- prep_roc(base_data_match_without_uti, thresh)
  roc_obj1 <- roc(roc_with$actual, roc_with$predicted, quiet = TRUE)
  roc_obj2 <- roc(roc_without$actual, roc_without$predicted, quiet = TRUE)
  test_result <- roc.test(roc_obj1, roc_obj2, method = "delong")
  p_val <- test_result$p.value
  p_label <- if_else(p_val < 0.001, "<0.001", sprintf("%.3f", p_val))
  results_matched_df <- rbind(results_matched_df,
    extract_metrics(roc_obj2, "Without UTI", labels[i], p_label),
    extract_metrics(roc_obj1, "With UTI", labels[i], ""))
}

results_matched_df

## ROC Curves

### Without UTI - Unmatched

In [ ]:
%%R
generate_roc_plot(base_data_without_uti, "ROC Curves: AoU EAU Risk Stratification Without rUTIs (Unmatched)")

### Without UTI - Matched

In [ ]:
%%R
generate_roc_plot(base_data_match_without_uti, "ROC Curves: AoU EAU Risk Stratification Without rUTIs (Matched)")

### Without UTI - Faceted by Initial Presentation

In [ ]:
%%R
generate_faceted_roc_plot(base_data_without_uti, "ROC Curves: AoU EAU Risk by Initial Presentation Without rUTIs")

### With UTI - Unmatched

In [ ]:
%%R
generate_roc_plot(base_data_with_uti, "ROC Curves: AoU EAU Risk Stratification With rUTIs (Unmatched)")

### With UTI - Matched

In [ ]:
%%R
generate_roc_plot(base_data_match_with_uti, "ROC Curves: AoU EAU Risk Stratification With rUTIs (Matched)")

### With UTI - Faceted by Initial Presentation

In [ ]:
%%R
generate_faceted_roc_plot(base_data_with_uti, "ROC Curves: AoU EAU Risk by Initial Presentation With rUTIs")

## Kaplan-Meier Plots

### Without UTI - Matched KM Dataset

In [ ]:
%%R
base_data_without_uti_for_km1 <- base_data_without_uti_for_km %>%
  mutate(high_risk_binary = if_else(high_risk == "high_risk", 1L, 0L),
         age_at_first_stone = as.numeric(age_at_first_stone),
         initial_presentation = as.factor(initial_presentation)) %>%
  drop_na(high_risk_binary, age_at_first_stone, initial_presentation)

m <- matchit(high_risk_binary ~ age_at_first_stone + initial_presentation,
             data = base_data_without_uti_for_km1, method = "nearest")
base_data_match_without_uti_match <- match.data(m)

### Without UTI - Unmatched KM

In [ ]:
%%R
km_fit <- surv_fit(Surv(time_to_recurrence_1, recurrence) ~ high_risk,
                   data = base_data_without_uti_for_km)

ggsurvplot(km_fit, data = base_data_without_uti_for_km,
  censor = FALSE, risk.table = TRUE, surv.median.line = "hv",
  conf.int = TRUE, tables.theme = theme_cleantable(),
  break.time.by = 1, xlab = "Time (years)", xlim = c(0, 10),
  title = "AoU: Time to First Recurrence Without rUTIs (Unmatched)", pval = TRUE,
  legend.title = "Risk Status", legend.labs = c("High Risk", "Low Risk"),
  ggtheme = theme_bw())

### Without UTI - Matched KM

In [ ]:
%%R
km_fit <- surv_fit(Surv(time_to_recurrence_1, recurrence) ~ high_risk,
                   data = base_data_match_without_uti_match)

ggsurvplot(km_fit, data = base_data_match_without_uti_match,
  censor = FALSE, risk.table = TRUE, surv.median.line = "hv",
  conf.int = TRUE, tables.theme = theme_cleantable(),
  break.time.by = 1, xlab = "Time (years)", xlim = c(0, 10),
  title = "AoU: Time to First Recurrence Without rUTIs (Matched)", pval = TRUE,
  legend.title = "Risk Status", legend.labs = c("High Risk", "Low Risk"),
  ggtheme = theme_bw())

### Without UTI - Faceted KM

In [ ]:
%%R
km_fit <- surv_fit(Surv(time_to_recurrence_1, recurrence) ~ high_risk,
                   data = as.data.frame(base_data_without_uti_for_km))

ggsurvplot_facet(km_fit,
  data = as.data.frame(base_data_without_uti_for_km),
  facet.by = "initial_presentation",
  censor = FALSE, conf.int = TRUE, surv.median.line = "hv",
  pval = TRUE, break.time.by = 1, xlim = c(0, 10),
  xlab = "Time (years)", ylab = "Recurrence-free probability",
  legend.title = "Risk Status", legend.labs = c("High Risk", "Low Risk"),
  ggtheme = theme_bw(), panel.labs.font = list(face = "bold"))

### With UTI - Matched KM Dataset

In [ ]:
%%R
base_data_with_uti_for_km1 <- base_data_with_uti_for_km %>%
  mutate(high_risk_binary = if_else(high_risk == "high_risk", 1L, 0L),
         age_at_first_stone = as.numeric(age_at_first_stone),
         initial_presentation = as.factor(initial_presentation)) %>%
  drop_na(high_risk_binary, age_at_first_stone, initial_presentation)

m <- matchit(high_risk_binary ~ age_at_first_stone + initial_presentation,
             data = base_data_with_uti_for_km1, method = "nearest")
base_data_match_with_uti_match <- match.data(m)

### With UTI - Unmatched KM

In [ ]:
%%R
km_fit <- surv_fit(Surv(time_to_recurrence_1, recurrence) ~ high_risk,
                   data = base_data_with_uti_for_km)

ggsurvplot(km_fit, data = base_data_with_uti_for_km,
  censor = FALSE, risk.table = TRUE, surv.median.line = "hv",
  conf.int = TRUE, tables.theme = theme_cleantable(),
  break.time.by = 1, xlab = "Time (years)", xlim = c(0, 10),
  title = "AoU: Time to First Recurrence With rUTIs (Unmatched)", pval = TRUE,
  legend.title = "Risk Status", legend.labs = c("High Risk", "Low Risk"),
  ggtheme = theme_bw())

### With UTI - Matched KM

In [ ]:
%%R
km_fit <- surv_fit(Surv(time_to_recurrence_1, recurrence) ~ high_risk,
                   data = base_data_match_with_uti_match)

ggsurvplot(km_fit, data = base_data_match_with_uti_match,
  censor = FALSE, risk.table = TRUE, surv.median.line = "hv",
  conf.int = TRUE, tables.theme = theme_cleantable(),
  break.time.by = 1, xlab = "Time (years)", xlim = c(0, 10),
  title = "AoU: Time to First Recurrence With rUTIs (Matched)", pval = TRUE,
  legend.title = "Risk Status", legend.labs = c("High Risk", "Low Risk"),
  ggtheme = theme_bw())

### With UTI - Faceted KM

In [ ]:
%%R
km_fit <- surv_fit(Surv(time_to_recurrence_1, recurrence) ~ high_risk,
                   data = as.data.frame(base_data_with_uti_for_km))

ggsurvplot_facet(km_fit,
  data = as.data.frame(base_data_with_uti_for_km),
  facet.by = "initial_presentation",
  censor = FALSE, conf.int = TRUE, surv.median.line = "hv",
  pval = TRUE, break.time.by = 1, xlim = c(0, 10),
  xlab = "Time (years)", ylab = "Recurrence-free probability",
  legend.title = "Risk Status", legend.labs = c("High Risk", "Low Risk"),
  ggtheme = theme_bw(), panel.labs.font = list(face = "bold"))

## Decision Curve Analysis (DCA)

### Without UTI - Unmatched DCA

In [ ]:
%%R

thresholds <- c(1, 2, 3)
labels <- c("0 vs ≥1", "0 vs ≥2", "0 vs ≥3")

dca_list <- lapply(seq_along(thresholds), function(i) {
  thresh <- thresholds[i]
  dca_data <- base_data_without_uti %>%
    mutate(
      actual = case_when(count_diff >= thresh ~ 1, count_diff == 0 ~ 0, TRUE ~ NA_integer_),
      high_risk = case_when(high_risk == "low_risk" ~ 0, high_risk == "high_risk" ~ 1, TRUE ~ NA_integer_)
    ) %>%
    drop_na(actual, high_risk)
  dca(actual ~ high_risk, data = dca_data, thresholds = seq(0, 0.8, by = 0.01))
})

combined_df <- lapply(seq_along(dca_list), function(i) {
  p <- plot(dca_list[[i]])
  p$data %>% mutate(comparison = labels[i])
}) %>% bind_rows()

ggplot(combined_df, aes(x = threshold, y = net_benefit, colour = label)) +
  geom_line(linewidth = 0.8) +
  facet_wrap(~ comparison, ncol = 3) +
  labs(x = "Probability of Recurrence Threshold", y = "Net Benefit",
       colour = NULL, title = "DCA: AoU Unmatched Dataset Without rUTIs") +
  theme_bw(base_size = 12) + theme(legend.position = "bottom") +
  scale_colour_discrete(labels = c("Follow-up All", "Follow-up None", "Risk Status")) +
  xlim(c(0, 0.6)) + ylim(c(-1.5, 0.5)) + geom_vline(xintercept = 0.2, linetype = "dashed")

### Without UTI - Matched DCA

In [ ]:
%%R

thresholds <- c(1, 2, 3)
labels <- c("0 vs ≥1", "0 vs ≥2", "0 vs ≥3")

dca_list <- lapply(seq_along(thresholds), function(i) {
  thresh <- thresholds[i]
  dca_data <- base_data_match_without_uti %>%
    mutate(
      actual = case_when(count_diff >= thresh ~ 1, count_diff == 0 ~ 0, TRUE ~ NA_integer_),
      high_risk = case_when(high_risk == "low_risk" ~ 0, high_risk == "high_risk" ~ 1, TRUE ~ NA_integer_)
    ) %>%
    drop_na(actual, high_risk)
  dca(actual ~ high_risk, data = dca_data, thresholds = seq(0, 0.8, by = 0.01))
})

combined_df <- lapply(seq_along(dca_list), function(i) {
  p <- plot(dca_list[[i]])
  p$data %>% mutate(comparison = labels[i])
}) %>% bind_rows()

ggplot(combined_df, aes(x = threshold, y = net_benefit, colour = label)) +
  geom_line(linewidth = 0.8) +
  facet_wrap(~ comparison, ncol = 3) +
  labs(x = "Probability of Recurrence Threshold", y = "Net Benefit",
       colour = NULL, title = "DCA: AoU Matched Dataset Without rUTIs") +
  theme_bw(base_size = 12) + theme(legend.position = "bottom") +
  scale_colour_discrete(labels = c("Follow-up All", "Follow-up None", "Risk Status")) +
  xlim(c(0, 0.6)) + ylim(c(-1.5, 0.5)) + geom_vline(xintercept = 0.2, linetype = "dashed")

### Without UTI - DCA by Initial Presentation

In [ ]:
%%R

thresholds <- c(1, 2, 3)
labels <- c("0 vs ≥1", "0 vs ≥2", "0 vs ≥3")
presentations <- unique(base_data_without_uti$initial_presentation)

combined_df <- lapply(presentations, function(pres) {
  lapply(seq_along(thresholds), function(i) {
    thresh <- thresholds[i]
    dca_data <- base_data_without_uti %>%
      filter(initial_presentation == pres) %>%
      mutate(
        actual = case_when(count_diff >= thresh ~ 1, count_diff == 0 ~ 0, TRUE ~ NA_integer_),
        high_risk = case_when(high_risk == "low_risk" ~ 0, high_risk == "high_risk" ~ 1, TRUE ~ NA_integer_)
      ) %>%
      drop_na(actual, high_risk)
    if (nrow(dca_data) < 10 || length(unique(dca_data$actual)) < 2) return(NULL)
    dca_obj <- dca(actual ~ high_risk, data = dca_data, thresholds = seq(0, 0.8, by = 0.01))
    p <- plot(dca_obj)
    p$data %>% mutate(comparison = labels[i], initial_presentation = pres)
  }) %>% bind_rows()
}) %>% bind_rows()

ggplot(combined_df, aes(x = threshold, y = net_benefit, colour = label)) +
  geom_line(linewidth = 0.8) +
  facet_grid(initial_presentation ~ comparison) +
  labs(x = "Probability of Recurrence Threshold", y = "Net Benefit",
       colour = NULL, title = "DCA: AoU Without rUTIs by Initial Presentation") +
  theme_bw(base_size = 12) + theme(legend.position = "bottom") +
  scale_colour_discrete(labels = c("Follow-up All", "Follow-up None", "Risk Status")) +
  xlim(c(0, 0.4)) + ylim(c(-1.0, 0.6)) + geom_vline(xintercept = 0.2, linetype = "dashed")

### With UTI - Unmatched DCA

In [ ]:
%%R

thresholds <- c(1, 2, 3)
labels <- c("0 vs ≥1", "0 vs ≥2", "0 vs ≥3")

dca_list <- lapply(seq_along(thresholds), function(i) {
  thresh <- thresholds[i]
  dca_data <- base_data_with_uti %>%
    mutate(
      actual = case_when(count_diff >= thresh ~ 1, count_diff == 0 ~ 0, TRUE ~ NA_integer_),
      high_risk = case_when(high_risk == "low_risk" ~ 0, high_risk == "high_risk" ~ 1, TRUE ~ NA_integer_)
    ) %>%
    drop_na(actual, high_risk)
  dca(actual ~ high_risk, data = dca_data, thresholds = seq(0, 0.8, by = 0.01))
})

combined_df <- lapply(seq_along(dca_list), function(i) {
  p <- plot(dca_list[[i]])
  p$data %>% mutate(comparison = labels[i])
}) %>% bind_rows()

ggplot(combined_df, aes(x = threshold, y = net_benefit, colour = label)) +
  geom_line(linewidth = 0.8) +
  facet_wrap(~ comparison, ncol = 3) +
  labs(x = "Probability of Recurrence Threshold", y = "Net Benefit",
       colour = NULL, title = "DCA: AoU Unmatched Dataset With rUTIs") +
  theme_bw(base_size = 12) + theme(legend.position = "bottom") +
  scale_colour_discrete(labels = c("Follow-up All", "Follow-up None", "Risk Status")) +
  xlim(c(0, 0.6)) + ylim(c(-1.5, 0.5)) + geom_vline(xintercept = 0.2, linetype = "dashed")

### With UTI - Matched DCA

In [ ]:
%%R

thresholds <- c(1, 2, 3)
labels <- c("0 vs ≥1", "0 vs ≥2", "0 vs ≥3")

dca_list <- lapply(seq_along(thresholds), function(i) {
  thresh <- thresholds[i]
  dca_data <- base_data_match_with_uti %>%
    mutate(
      actual = case_when(count_diff >= thresh ~ 1, count_diff == 0 ~ 0, TRUE ~ NA_integer_),
      high_risk = case_when(high_risk == "low_risk" ~ 0, high_risk == "high_risk" ~ 1, TRUE ~ NA_integer_)
    ) %>%
    drop_na(actual, high_risk)
  dca(actual ~ high_risk, data = dca_data, thresholds = seq(0, 0.8, by = 0.01))
})

combined_df <- lapply(seq_along(dca_list), function(i) {
  p <- plot(dca_list[[i]])
  p$data %>% mutate(comparison = labels[i])
}) %>% bind_rows()

ggplot(combined_df, aes(x = threshold, y = net_benefit, colour = label)) +
  geom_line(linewidth = 0.8) +
  facet_wrap(~ comparison, ncol = 3) +
  labs(x = "Probability of Recurrence Threshold", y = "Net Benefit",
       colour = NULL, title = "DCA: AoU Matched Dataset With rUTIs") +
  theme_bw(base_size = 12) + theme(legend.position = "bottom") +
  scale_colour_discrete(labels = c("Follow-up All", "Follow-up None", "Risk Status")) +
  xlim(c(0, 0.6)) + ylim(c(-1.5, 0.5)) + geom_vline(xintercept = 0.2, linetype = "dashed")

### With UTI - DCA by Initial Presentation

In [ ]:
%%R

thresholds <- c(1, 2, 3)
labels <- c("0 vs ≥1", "0 vs ≥2", "0 vs ≥3")
presentations <- unique(base_data_with_uti$initial_presentation)

combined_df <- lapply(presentations, function(pres) {
  lapply(seq_along(thresholds), function(i) {
    thresh <- thresholds[i]
    dca_data <- base_data_with_uti %>%
      filter(initial_presentation == pres) %>%
      mutate(
        actual = case_when(count_diff >= thresh ~ 1, count_diff == 0 ~ 0, TRUE ~ NA_integer_),
        high_risk = case_when(high_risk == "low_risk" ~ 0, high_risk == "high_risk" ~ 1, TRUE ~ NA_integer_)
      ) %>%
      drop_na(actual, high_risk)
    if (nrow(dca_data) < 10 || length(unique(dca_data$actual)) < 2) return(NULL)
    dca_obj <- dca(actual ~ high_risk, data = dca_data, thresholds = seq(0, 0.8, by = 0.01))
    p <- plot(dca_obj)
    p$data %>% mutate(comparison = labels[i], initial_presentation = pres)
  }) %>% bind_rows()
}) %>% bind_rows()

ggplot(combined_df, aes(x = threshold, y = net_benefit, colour = label)) +
  geom_line(linewidth = 0.8) +
  facet_grid(initial_presentation ~ comparison) +
  labs(x = "Probability of Recurrence Threshold", y = "Net Benefit",
       colour = NULL, title = "DCA: AoU With rUTIs by Initial Presentation") +
  theme_bw(base_size = 12) + theme(legend.position = "bottom") +
  scale_colour_discrete(labels = c("Follow-up All", "Follow-up None", "Risk Status")) +
  xlim(c(0, 0.4)) + ylim(c(-1.0, 0.6)) + geom_vline(xintercept = 0.2, linetype = "dashed")

# Prognostic Accuracy Grid Search

Replicates `prognostic_accuracy_grid_search.Rmd` for AoU data.

## Grid Search Configuration

In [ ]:
%%R

starting_points <- c("index_date", "first_date", "second_date", "third_date")
recurrence_thresholds <- c(1, 2, 3)
recurrence_labels <- c("\u22651 subsequent stone events", "\u22652 subsequent stone events", "\u22653 subsequent stone events")
time_cutoffs_months <- seq(6, 24, by = 3)
follow_up_years <- 5
colors_rec <- c("#1b9e77", "#d95f02", "#7570b3")

## Build Analysis Dataset Function

In [ ]:
%%R

build_analysis_dataset <- function(starting_point_col,
                                   inter_event_cutoff_months = NULL,
                                   include_uti = FALSE) {
  ds <- dates_detailed %>%
    dplyr::select(participant_id, index_date, first_date, second_date, third_date,
                  fourth_date, fifth_date, n_recurrences, mean_time_between,
                  gap_index_to_first_months, gap_first_to_second_months,
                  gap_second_to_third_months) %>%
    mutate(start_date = !!sym(starting_point_col))

  ds <- ds %>% filter(!is.na(start_date))

  ds <- ds %>%
    mutate(
      subsequent_events = case_when(
        starting_point_col == "index_date" ~ n_recurrences,
        starting_point_col == "first_date" ~ pmax(n_recurrences - 1L, 0L),
        starting_point_col == "second_date" ~ pmax(n_recurrences - 2L, 0L),
        starting_point_col == "third_date" ~ pmax(n_recurrences - 3L, 0L)
      )
    )

  last_date_df <- overall_follow_up %>% dplyr::select(participant_id, last_date)

  ds <- ds %>%
    left_join(last_date_df, by = "participant_id") %>%
    drop_na(last_date)

  ds <- ds %>%
    mutate(
      next_event_date = case_when(
        starting_point_col == "index_date" ~ first_date,
        starting_point_col == "first_date" ~ second_date,
        starting_point_col == "second_date" ~ third_date,
        starting_point_col == "third_date" ~ fourth_date
      ),
      has_next_event = as.integer(!is.na(next_event_date)),
      time_to_next_years = if_else(
        has_next_event == 1L,
        as.numeric(next_event_date - start_date) / 365.25,
        as.numeric(last_date - start_date) / 365.25
      ),
      # Follow-up from the starting point (used to exclude ambiguous non-recurrers)
      follow_up_from_start_years = as.numeric(last_date - start_date) / 365.25
    ) %>%
    filter(time_to_next_years > 0)

  hr_df <- build_high_risk_df(starting_point_col, include_uti = include_uti)
  ds <- ds %>%
    left_join(hr_df, by = "participant_id") %>%
    left_join(age_at_first_stone_df, by = "participant_id")

  ds <- ds %>%
    left_join(first_presentation %>% dplyr::select(participant_id, initial_presentation),
              by = "participant_id")

  if (!is.null(inter_event_cutoff_months) && starting_point_col != "index_date") {
    gap_col <- case_when(
      starting_point_col == "first_date"  ~ "gap_index_to_first_months",
      starting_point_col == "second_date" ~ "gap_first_to_second_months",
      starting_point_col == "third_date"  ~ "gap_second_to_third_months"
    )
    ds <- ds %>%
      mutate(
        prior_gap_months = !!sym(gap_col),
        high_risk_with_time = case_when(
          high_risk == "high_risk" ~ "high_risk",
          !is.na(prior_gap_months) & prior_gap_months <= inter_event_cutoff_months ~ "high_risk",
          TRUE ~ "low_risk"
        )
      )
  } else {
    ds <- ds %>%
      mutate(high_risk_with_time = high_risk)
  }

  return(ds)
}

## Grid Search Helper Functions

In [ ]:
%%R

# Binary classification: exclude non-recurrers who don't have enough follow-up
# - Recurrers (subsequent_events >= thresh): always included (actual = 1)
# - Non-recurrers WITH >= follow_up_years follow-up: included (actual = 0)
# - Non-recurrers WITHOUT follow_up_years follow-up: EXCLUDED (ambiguous)
prep_roc_grid <- function(ds, thresh, risk_col = "high_risk_with_time") {
  ds %>%
    mutate(
      actual = case_when(
        subsequent_events >= thresh ~ 1L,
        subsequent_events < thresh & follow_up_from_start_years >= follow_up_years ~ 0L,
        TRUE ~ NA_integer_
      ),
      predicted = if_else(!!sym(risk_col) == "high_risk", 1, 0)
    ) %>%
    drop_na(actual, predicted)
}

extract_metrics_grid <- function(roc_obj, label, roc_data = NULL) {
  auc_val <- as.numeric(auc(roc_obj))
  auc_ci  <- ci.auc(roc_obj)
  coords_best <- coords(roc_obj, "best", ret = c("sensitivity", "specificity"), best.method = "youden")

  tp <- fp <- tn <- fn <- NA_integer_
  ppv_val <- npv_val <- sens_val <- spec_val <- NA_real_
  ppv_lo <- ppv_hi <- npv_lo <- npv_hi <- NA_real_
  sens_lo <- sens_hi <- spec_lo <- spec_hi <- NA_real_
  n_total <- n_high_risk <- n_low_risk <- NA_integer_
  n_recurred_high <- n_recurred_low <- NA_integer_
  pct_recurred_high <- pct_recurred_low <- NA_real_
  pct_recurred_high_lo <- pct_recurred_high_hi <- NA_real_
  pct_recurred_low_lo  <- pct_recurred_low_hi  <- NA_real_

  sens_val <- coords_best$sensitivity
  spec_val <- coords_best$specificity

  if (!is.null(roc_data)) {
    tp <- sum(roc_data$predicted == 1 & roc_data$actual == 1)
    fp <- sum(roc_data$predicted == 1 & roc_data$actual == 0)
    tn <- sum(roc_data$predicted == 0 & roc_data$actual == 0)
    fn <- sum(roc_data$predicted == 0 & roc_data$actual == 1)
    n_total <- nrow(roc_data)

    ppv_val <- tp / (tp + fp)
    npv_val <- tn / (tn + fn)

    wilson_ci <- function(x, n) {
      if (n == 0) return(c(NA_real_, NA_real_))
      bt <- binom.test(x, n)
      as.numeric(bt$conf.int)
    }

    sens_ci <- wilson_ci(tp, tp + fn)
    sens_lo <- sens_ci[1]; sens_hi <- sens_ci[2]
    spec_ci <- wilson_ci(tn, tn + fp)
    spec_lo <- spec_ci[1]; spec_hi <- spec_ci[2]
    ppv_ci <- wilson_ci(tp, tp + fp)
    ppv_lo <- ppv_ci[1]; ppv_hi <- ppv_ci[2]
    npv_ci <- wilson_ci(tn, tn + fn)
    npv_lo <- npv_ci[1]; npv_hi <- npv_ci[2]

    n_high_risk     <- tp + fp
    n_low_risk      <- tn + fn
    n_recurred_high <- tp
    n_recurred_low  <- fn

    pct_recurred_high <- if (n_high_risk > 0) tp / n_high_risk * 100 else NA_real_
    pct_recurred_low  <- if (n_low_risk > 0)  fn / n_low_risk  * 100 else NA_real_

    if (n_high_risk > 0) {
      hr_ci <- wilson_ci(tp, n_high_risk)
      pct_recurred_high_lo <- hr_ci[1] * 100; pct_recurred_high_hi <- hr_ci[2] * 100
    }
    if (n_low_risk > 0) {
      lr_ci <- wilson_ci(fn, n_low_risk)
      pct_recurred_low_lo <- lr_ci[1] * 100; pct_recurred_low_hi <- lr_ci[2] * 100
    }
  }

  data.frame(
    Comparison = label,
    N = n_total, N_High_Risk = n_high_risk, N_Low_Risk = n_low_risk,
    AUC = sprintf("%.3f (%.3f\u2013%.3f)", auc_val, auc_ci[1], auc_ci[3]),
    AUC_val = auc_val, AUC_lo = as.numeric(auc_ci[1]), AUC_hi = as.numeric(auc_ci[3]),
    Sensitivity = sprintf("%.3f (%.3f\u2013%.3f)", sens_val, sens_lo, sens_hi),
    Sensitivity_val = sens_val, Sensitivity_lo = sens_lo, Sensitivity_hi = sens_hi,
    Specificity = sprintf("%.3f (%.3f\u2013%.3f)", spec_val, spec_lo, spec_hi),
    Specificity_val = spec_val, Specificity_lo = spec_lo, Specificity_hi = spec_hi,
    PPV = sprintf("%.3f (%.3f\u2013%.3f)", ppv_val, ppv_lo, ppv_hi),
    PPV_val = ppv_val, PPV_lo = ppv_lo, PPV_hi = ppv_hi,
    NPV = sprintf("%.3f (%.3f\u2013%.3f)", npv_val, npv_lo, npv_hi),
    NPV_val = npv_val, NPV_lo = npv_lo, NPV_hi = npv_hi,
    Likelihood_Recurrence_High_Risk_pct = pct_recurred_high,
    Likelihood_Recurrence_High_Risk_lo = pct_recurred_high_lo,
    Likelihood_Recurrence_High_Risk_hi = pct_recurred_high_hi,
    N_Recurred_High_Risk = n_recurred_high,
    Likelihood_Recurrence_Low_Risk_pct = pct_recurred_low,
    Likelihood_Recurrence_Low_Risk_lo = pct_recurred_low_lo,
    Likelihood_Recurrence_Low_Risk_hi = pct_recurred_low_hi,
    N_Recurred_Low_Risk = n_recurred_low,
    stringsAsFactors = FALSE
  )
}

generate_roc_plot_grid <- function(ds, plot_title, risk_col = "high_risk_with_time") {
  roc_combined_df <- data.frame()
  ci_combined_df <- data.frame()
  auc_table_df <- data.frame()
  for (i in seq_along(recurrence_thresholds)) {
    thresh <- recurrence_thresholds[i]
    label <- recurrence_labels[i]
    roc_data <- prep_roc_grid(ds, thresh, risk_col)
    if (nrow(roc_data) < 10 || length(unique(roc_data$actual)) < 2) next
    roc_obj <- roc(roc_data$actual, roc_data$predicted, quiet = TRUE)
    roc_combined_df <- rbind(roc_combined_df, data.frame(
      fpr = 1 - roc_obj$specificities, tpr = roc_obj$sensitivities, Comparison = label))
    ci_se <- ci.se(roc_obj, specificities = seq(0, 1, length.out = 100))
    ci_combined_df <- rbind(ci_combined_df, data.frame(
      fpr = 1 - as.numeric(rownames(ci_se)),
      tpr_lower = ci_se[, 1], tpr_upper = ci_se[, 3], Comparison = label))
    auc_val <- as.numeric(auc(roc_obj))
    auc_ci <- ci.auc(roc_obj)
    auc_table_df <- rbind(auc_table_df, data.frame(
      Comparison = label,
      AUC_CI = sprintf("%.2f (%.2f\u2013%.2f)", auc_val, auc_ci[1], auc_ci[3])))
  }
  if (nrow(roc_combined_df) == 0) return(NULL)
  auc_table <- tableGrob(auc_table_df, rows = NULL, theme = ttheme_minimal(base_size = 9))
  auc_table_boxed <- grobTree(
    rectGrob(gp = gpar(col = "black", fill = "white", lwd = 1)), auc_table)
  ggplot() +
    geom_ribbon(data = ci_combined_df,
                aes(x = fpr, ymin = tpr_lower, ymax = tpr_upper, fill = Comparison), alpha = 0.2) +
    geom_line(data = roc_combined_df,
              aes(x = fpr, y = tpr, color = Comparison), linewidth = 1) +
    geom_abline(linetype = "dashed", alpha = 0.6) +
    labs(x = "False Positive Rate", y = "True Positive Rate", title = plot_title) +
    scale_color_manual(values = colors_rec) + scale_fill_manual(values = colors_rec) +
    coord_equal() + theme_bw(base_size = 12) + theme(legend.position = "bottom") +
    annotation_custom(grob = auc_table_boxed,
                      xmin = 0.55, xmax = 1, ymin = 0.05, ymax = 0.3)
}

generate_km_plot_grid <- function(ds, plot_title, risk_col = "high_risk_with_time") {
  km_data <- ds %>%
    rename(risk_group = !!sym(risk_col)) %>%
    filter(!is.na(time_to_next_years), !is.na(risk_group)) %>%
    mutate(recurrence = has_next_event)
  if (nrow(km_data) < 10 || length(unique(km_data$risk_group)) < 2) return(NULL)
  km_fit <- surv_fit(Surv(time_to_next_years, recurrence) ~ risk_group, data = km_data)
  ggsurvplot(km_fit, data = km_data,
    censor = FALSE, risk.table = TRUE, surv.median.line = "hv",
    conf.int = TRUE, tables.theme = theme_cleantable(),
    break.time.by = 1, xlab = "Time (years)", xlim = c(0, 10),
    title = plot_title, pval = TRUE,
    legend.title = "Risk Status", legend.labs = c("High Risk", "Low Risk"),
    ggtheme = theme_bw())
}

generate_dca_plot_grid <- function(ds, plot_title, risk_col = "high_risk_with_time") {
  dca_list <- lapply(seq_along(recurrence_thresholds), function(i) {
    thresh <- recurrence_thresholds[i]
    dca_data <- ds %>%
      mutate(
        actual = case_when(
          subsequent_events >= thresh ~ 1L,
          subsequent_events < thresh & follow_up_from_start_years >= follow_up_years ~ 0L,
          TRUE ~ NA_integer_
        ),
        risk_binary = case_when(
          !!sym(risk_col) == "low_risk" ~ 0, !!sym(risk_col) == "high_risk" ~ 1, TRUE ~ NA_real_)
      ) %>%
      drop_na(actual, risk_binary)
    if (nrow(dca_data) < 10 || length(unique(dca_data$actual)) < 2) return(NULL)
    dca(actual ~ risk_binary, data = dca_data, thresholds = seq(0, 0.8, by = 0.01))
  })
  dca_list <- dca_list[!sapply(dca_list, is.null)]
  if (length(dca_list) == 0) return(NULL)
  labels_used <- recurrence_labels[seq_along(dca_list)]
  combined_df <- lapply(seq_along(dca_list), function(i) {
    p <- plot(dca_list[[i]])
    p$data %>% mutate(comparison = labels_used[i])
  }) %>% bind_rows()
  ggplot(combined_df, aes(x = threshold, y = net_benefit, colour = label)) +
    geom_line(linewidth = 0.8) +
    facet_wrap(~ comparison, ncol = 3) +
    labs(x = "Probability of Recurrence Threshold", y = "Net Benefit",
         colour = NULL, title = plot_title) +
    theme_bw(base_size = 12) + theme(legend.position = "bottom") +
    scale_colour_discrete(labels = c("Follow-up All", "Follow-up None", "Risk Status")) +
    xlim(c(0, 0.6)) + geom_vline(xintercept = 0.2, linetype = "dashed")
}

## Analysis 1: Index Date (Without UTI)

In [ ]:
%%R
ds_index <- build_analysis_dataset("index_date", include_uti = FALSE)
cat("N participants from index_date (without UTI):", nrow(ds_index), "\n")
table(ds_index$subsequent_events)

In [ ]:
%%R
roc_index <- generate_roc_plot_grid(ds_index, "ROC: From Index Date (EAU Risk, Without UTI)")
roc_index

In [ ]:
%%R
km_index <- generate_km_plot_grid(ds_index, "KM: From Index Date (EAU Risk, Without UTI)")
km_index

In [ ]:
%%R
dca_index <- generate_dca_plot_grid(ds_index, "DCA: From Index Date (EAU Risk, Without UTI)")
dca_index

In [ ]:
%%R
metrics_index_no_uti <- lapply(seq_along(recurrence_thresholds), function(i) {
  roc_data <- prep_roc_grid(ds_index, recurrence_thresholds[i])
  if (nrow(roc_data) < 10 || length(unique(roc_data$actual)) < 2) return(NULL)
  roc_obj <- roc(roc_data$actual, roc_data$predicted, quiet = TRUE)
  extract_metrics_grid(roc_obj, recurrence_labels[i], roc_data) %>%
    mutate(Starting_Point = "Index Date", Time_Cutoff_Months = NA_real_, UTI = "Without")
}) %>% bind_rows()

metrics_index_no_uti

## Analysis 1: Index Date (With UTI)

In [ ]:
%%R
ds_index_uti <- build_analysis_dataset("index_date", include_uti = TRUE)
cat("N participants from index_date (with UTI):", nrow(ds_index_uti), "\n")
table(ds_index_uti$subsequent_events)

In [ ]:
%%R
roc_index_uti <- generate_roc_plot_grid(ds_index_uti, "ROC: From Index Date (EAU Risk, With UTI)")
roc_index_uti

In [ ]:
%%R
km_index_uti <- generate_km_plot_grid(ds_index_uti, "KM: From Index Date (EAU Risk, With UTI)")
km_index_uti

In [ ]:
%%R
dca_index_uti <- generate_dca_plot_grid(ds_index_uti, "DCA: From Index Date (EAU Risk, With UTI)")
dca_index_uti

In [ ]:
%%R
metrics_index_with_uti <- lapply(seq_along(recurrence_thresholds), function(i) {
  roc_data <- prep_roc_grid(ds_index_uti, recurrence_thresholds[i])
  if (nrow(roc_data) < 10 || length(unique(roc_data$actual)) < 2) return(NULL)
  roc_obj <- roc(roc_data$actual, roc_data$predicted, quiet = TRUE)
  extract_metrics_grid(roc_obj, recurrence_labels[i], roc_data) %>%
    mutate(Starting_Point = "Index Date", Time_Cutoff_Months = NA_real_, UTI = "With")
}) %>% bind_rows()

metrics_index_with_uti

## Grid Search Function

In [ ]:
%%R

run_grid_search <- function(starting_point_col, sp_label, include_uti = FALSE) {
  all_metrics <- data.frame()
  all_roc_plots <- list()
  all_km_plots <- list()
  all_dca_plots <- list()

  for (cutoff in time_cutoffs_months) {
    ds <- build_analysis_dataset(starting_point_col,
                                 inter_event_cutoff_months = cutoff,
                                 include_uti = include_uti)
    if (nrow(ds) < 20) next

    uti_tag <- if (include_uti) ", With UTI" else ", Without UTI"
    cutoff_label <- paste0(sp_label, " | Gap \u2264", cutoff, "mo = High Risk", uti_tag)

    for (i in seq_along(recurrence_thresholds)) {
      roc_data <- prep_roc_grid(ds, recurrence_thresholds[i])
      if (nrow(roc_data) < 10 || length(unique(roc_data$actual)) < 2) next
      roc_obj <- roc(roc_data$actual, roc_data$predicted, quiet = TRUE)
      m <- extract_metrics_grid(roc_obj, recurrence_labels[i], roc_data)
      m$Starting_Point <- sp_label
      m$Time_Cutoff_Months <- cutoff
      m$UTI <- if (include_uti) "With" else "Without"
      all_metrics <- rbind(all_metrics, m)
    }

    roc_p <- generate_roc_plot_grid(ds, paste0("ROC: ", cutoff_label))
    if (!is.null(roc_p)) all_roc_plots[[paste0("cutoff_", cutoff)]] <- roc_p
    km_p <- generate_km_plot_grid(ds, paste0("KM: ", cutoff_label))
    if (!is.null(km_p)) all_km_plots[[paste0("cutoff_", cutoff)]] <- km_p
    dca_p <- generate_dca_plot_grid(ds, paste0("DCA: ", cutoff_label))
    if (!is.null(dca_p)) all_dca_plots[[paste0("cutoff_", cutoff)]] <- dca_p
  }

  list(metrics = all_metrics, roc_plots = all_roc_plots,
       km_plots = all_km_plots, dca_plots = all_dca_plots)
}

## Analysis 2: From Second Event (First Date)

In [ ]:
%%R
results_first_no_uti <- run_grid_search("first_date", "Second Event", include_uti = FALSE)
if (nrow(results_first_no_uti$metrics) > 0) {
  results_first_no_uti$metrics
}

In [ ]:
%%R
for (nm in names(results_first_no_uti$roc_plots)) print(results_first_no_uti$roc_plots[[nm]])
for (nm in names(results_first_no_uti$km_plots)) print(results_first_no_uti$km_plots[[nm]])
for (nm in names(results_first_no_uti$dca_plots)) print(results_first_no_uti$dca_plots[[nm]])

In [ ]:
%%R
results_first_with_uti <- run_grid_search("first_date", "Second Event", include_uti = TRUE)
if (nrow(results_first_with_uti$metrics) > 0) {
  results_first_with_uti$metrics
}

In [ ]:
%%R
for (nm in names(results_first_with_uti$roc_plots)) print(results_first_with_uti$roc_plots[[nm]])
for (nm in names(results_first_with_uti$km_plots)) print(results_first_with_uti$km_plots[[nm]])
for (nm in names(results_first_with_uti$dca_plots)) print(results_first_with_uti$dca_plots[[nm]])

## Analysis 3: From Third Event (Second Date)

In [ ]:
%%R
results_second_no_uti <- run_grid_search("second_date", "Third Event", include_uti = FALSE)
if (nrow(results_second_no_uti$metrics) > 0) {
  results_second_no_uti$metrics
}

In [ ]:
%%R
for (nm in names(results_second_no_uti$roc_plots)) print(results_second_no_uti$roc_plots[[nm]])
for (nm in names(results_second_no_uti$km_plots)) print(results_second_no_uti$km_plots[[nm]])
for (nm in names(results_second_no_uti$dca_plots)) print(results_second_no_uti$dca_plots[[nm]])

In [ ]:
%%R
results_second_with_uti <- run_grid_search("second_date", "Third Event", include_uti = TRUE)
if (nrow(results_second_with_uti$metrics) > 0) {
  results_second_with_uti$metrics
}

In [ ]:
%%R
for (nm in names(results_second_with_uti$roc_plots)) print(results_second_with_uti$roc_plots[[nm]])
for (nm in names(results_second_with_uti$km_plots)) print(results_second_with_uti$km_plots[[nm]])
for (nm in names(results_second_with_uti$dca_plots)) print(results_second_with_uti$dca_plots[[nm]])

## Analysis 4: From Fourth Event (Third Date)

In [ ]:
%%R
results_third_no_uti <- run_grid_search("third_date", "Fourth Event", include_uti = FALSE)
if (nrow(results_third_no_uti$metrics) > 0) {
  results_third_no_uti$metrics
}

In [ ]:
%%R
for (nm in names(results_third_no_uti$roc_plots)) print(results_third_no_uti$roc_plots[[nm]])
for (nm in names(results_third_no_uti$km_plots)) print(results_third_no_uti$km_plots[[nm]])
for (nm in names(results_third_no_uti$dca_plots)) print(results_third_no_uti$dca_plots[[nm]])

In [ ]:
%%R
results_third_with_uti <- run_grid_search("third_date", "Fourth Event", include_uti = TRUE)
if (nrow(results_third_with_uti$metrics) > 0) {
  results_third_with_uti$metrics
}

In [ ]:
%%R
for (nm in names(results_third_with_uti$roc_plots)) print(results_third_with_uti$roc_plots[[nm]])
for (nm in names(results_third_with_uti$km_plots)) print(results_third_with_uti$km_plots[[nm]])
for (nm in names(results_third_with_uti$dca_plots)) print(results_third_with_uti$dca_plots[[nm]])

## Combined Metrics Summary

In [ ]:
%%R

all_metrics_combined <- bind_rows(
  metrics_index_no_uti, metrics_index_with_uti,
  results_first_no_uti$metrics, results_first_with_uti$metrics,
  results_second_no_uti$metrics, results_second_with_uti$metrics,
  results_third_no_uti$metrics, results_third_with_uti$metrics
)

all_metrics_combined %>%
  dplyr::select(Starting_Point, Time_Cutoff_Months, UTI, Comparison, N,
                AUC, Sensitivity, Specificity, PPV, NPV)

## Heatmaps

### AUC Heatmap

In [ ]:
%%R

heatmap_data <- all_metrics_combined %>%
  filter(!is.na(Time_Cutoff_Months), Starting_Point != "Index Date") %>%
  mutate(
    AUC_numeric = AUC_val,
    Time_Cutoff_label = paste0("\u2264", Time_Cutoff_Months, "mo")
  ) %>%
  mutate(
    Time_Cutoff_label = factor(Time_Cutoff_label,
      levels = c("\u22646mo", "\u22649mo", "\u226412mo", "\u226415mo", "\u226418mo", "\u226421mo", "\u226424mo")),
    Starting_Point = case_when(
      Starting_Point == "Second Event" ~ "Second Symptomatic Episode",
      Starting_Point == "Third Event" ~ "Third Symptomatic Episode",
      Starting_Point == "Fourth Event" ~ "Fourth Symptomatic Episode"
    ) %>% factor(levels = c("Fourth Symptomatic Episode",
                            "Third Symptomatic Episode",
                            "Second Symptomatic Episode")),
    UTI = case_when(UTI == "With" ~ "With UTI", UTI == "Without" ~ "Without UTI") %>%
      factor(levels = c("Without UTI", "With UTI")),
    Comparison = factor(Comparison, levels = c("\u22651 subsequent stone events",
                                               "\u22652 subsequent stone events",
                                               "\u22653 subsequent stone events"))
  )

median_auc <- median(heatmap_data$AUC_numeric)
min_auc <- min(heatmap_data$AUC_numeric)
max_auc <- max(heatmap_data$AUC_numeric)

auc_heatmap <- ggplot(heatmap_data,
       aes(x = Time_Cutoff_label, y = Starting_Point, fill = AUC_numeric)) +
  geom_tile(color = "white", linewidth = 0.5) +
  geom_text(aes(label = sprintf("%.2f", AUC_numeric)), size = 3) +
  facet_grid(UTI ~ Comparison) +
  scale_fill_gradient2(low = "red3", mid = "yellow", high = "#1a9850",
                       midpoint = median_auc, name = "AUC", limits = c(min_auc, max_auc)) +
  labs(x = "Threshold for 'Short Time' Definition",
       y = "Threshold for 'Recurrent' Definition",
       title = "AUC",
       subtitle = "            Subsequent Symptomatic Events needed for 'Recurrence' Definition") +
  theme_bw(base_size = 11) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        strip.text = element_text(face = "bold"), legend.position = "right")

auc_heatmap

### Sensitivity Heatmap

In [ ]:
%%R

median_sens <- median(heatmap_data$Sensitivity_val, na.rm = TRUE)
min_sens <- min(heatmap_data$Sensitivity_val, na.rm = TRUE)
max_sens <- max(heatmap_data$Sensitivity_val, na.rm = TRUE)

sens_heatmap <- ggplot(heatmap_data,
       aes(x = Time_Cutoff_label, y = Starting_Point, fill = Sensitivity_val)) +
  geom_tile(color = "white", linewidth = 0.5) +
  geom_text(aes(label = sprintf("%.2f", Sensitivity_val)), size = 3) +
  facet_grid(UTI ~ Comparison) +
  scale_fill_gradient2(low = "red3", mid = "yellow", high = "#1a9850",
                       midpoint = median_sens, name = "Sensitivity", limits = c(min_sens, max_sens)) +
  labs(x = "Threshold for 'Short Time' Definition",
       y = "Threshold for 'Recurrent' Definition",
       title = "Sensitivity",
       subtitle = "            Subsequent Symptomatic Events needed for 'Recurrence' Definition") +
  theme_bw(base_size = 11) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        strip.text = element_text(face = "bold"), legend.position = "right")

sens_heatmap

### Specificity Heatmap

In [ ]:
%%R

median_spec <- median(heatmap_data$Specificity_val, na.rm = TRUE)
min_spec <- min(heatmap_data$Specificity_val, na.rm = TRUE)
max_spec <- max(heatmap_data$Specificity_val, na.rm = TRUE)

spec_heatmap <- ggplot(heatmap_data,
       aes(x = Time_Cutoff_label, y = Starting_Point, fill = Specificity_val)) +
  geom_tile(color = "white", linewidth = 0.5) +
  geom_text(aes(label = sprintf("%.2f", Specificity_val)), size = 3) +
  facet_grid(UTI ~ Comparison) +
  scale_fill_gradient2(low = "red3", mid = "yellow", high = "#1a9850",
                       midpoint = median_spec, name = "Specificity", limits = c(min_spec, max_spec)) +
  labs(x = "Threshold for 'Short Time' Definition",
       y = "Threshold for 'Recurrent' Definition",
       title = "Specificity",
       subtitle = "            Subsequent Symptomatic Events needed for 'Recurrence' Definition") +
  theme_bw(base_size = 11) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        strip.text = element_text(face = "bold"), legend.position = "right")

spec_heatmap

### PPV Heatmap

In [ ]:
%%R

median_ppv <- median(heatmap_data$PPV_val, na.rm = TRUE)
min_ppv <- min(heatmap_data$PPV_val, na.rm = TRUE)
max_ppv <- max(heatmap_data$PPV_val, na.rm = TRUE)

ppv_heatmap <- ggplot(heatmap_data,
       aes(x = Time_Cutoff_label, y = Starting_Point, fill = PPV_val)) +
  geom_tile(color = "white", linewidth = 0.5) +
  geom_text(aes(label = sprintf("%.2f", PPV_val)), size = 3) +
  facet_grid(UTI ~ Comparison) +
  scale_fill_gradient2(low = "red3", mid = "yellow", high = "#1a9850",
                       midpoint = median_ppv, name = "PPV", limits = c(min_ppv, max_ppv)) +
  labs(x = "Threshold for 'Short Time' Definition",
       y = "Threshold for 'Recurrent' Definition",
       title = "Positive Predictive Value",
       subtitle = "            Subsequent Symptomatic Events needed for 'Recurrence' Definition") +
  theme_bw(base_size = 11) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        strip.text = element_text(face = "bold"), legend.position = "right")

ppv_heatmap

### NPV Heatmap

In [ ]:
%%R

median_npv <- median(heatmap_data$NPV_val, na.rm = TRUE)
min_npv <- min(heatmap_data$NPV_val, na.rm = TRUE)
max_npv <- max(heatmap_data$NPV_val, na.rm = TRUE)

npv_heatmap <- ggplot(heatmap_data,
       aes(x = Time_Cutoff_label, y = Starting_Point, fill = NPV_val)) +
  geom_tile(color = "white", linewidth = 0.5) +
  geom_text(aes(label = sprintf("%.2f", NPV_val)), size = 3) +
  facet_grid(UTI ~ Comparison) +
  scale_fill_gradient2(low = "red3", mid = "yellow", high = "#1a9850",
                       midpoint = median_npv, name = "NPV", limits = c(min_npv, max_npv)) +
  labs(x = "Threshold for 'Short Time' Definition",
       y = "Threshold for 'Recurrent' Definition",
       title = "Negative Predictive Value",
       subtitle = "            Subsequent Symptomatic Events needed for 'Recurrence' Definition") +
  theme_bw(base_size = 11) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        strip.text = element_text(face = "bold"), legend.position = "right")

npv_heatmap

### Likelihood of Recurrence Heatmaps

In [ ]:
%%R

likelihood_rows <- list()

for (sp_col in c("first_date", "second_date", "third_date")) {
  sp_label <- switch(sp_col,
                     first_date  = "Second Event",
                     second_date = "Third Event",
                     third_date  = "Fourth Event")

  for (cutoff in time_cutoffs_months) {
    for (uti_flag in c(FALSE, TRUE)) {
      ds <- build_analysis_dataset(sp_col,
                                   inter_event_cutoff_months = cutoff,
                                   include_uti = uti_flag)
      if (nrow(ds) < 20) next

      for (i in seq_along(recurrence_thresholds)) {
        thresh <- recurrence_thresholds[i]
        label  <- recurrence_labels[i]

        tmp <- ds %>%
          mutate(recurred = as.integer(subsequent_events >= thresh)) %>%
          group_by(high_risk_with_time) %>%
          summarise(n = n(),
                    n_recurred = sum(recurred, na.rm = TRUE),
                    .groups = "drop") %>%
          mutate(
            pct_recurred = n_recurred / n * 100,
            Starting_Point = sp_label,
            Time_Cutoff_Months = cutoff,
            Comparison = label,
            UTI = if (uti_flag) "With" else "Without"
          )
        likelihood_rows <- c(likelihood_rows, list(tmp))
      }
    }
  }
}

likelihood_df <- bind_rows(likelihood_rows) %>%
  mutate(
    Time_Cutoff_label = paste0("\u2264", Time_Cutoff_Months, "mo"),
    Time_Cutoff_label = factor(Time_Cutoff_label,
                               levels = c("\u22646mo", "\u22649mo", "\u226412mo",
                                          "\u226415mo", "\u226418mo", "\u226421mo",
                                          "\u226424mo")),
    Starting_Point = factor(Starting_Point,
                            levels = c("Fourth Event", "Third Event", "Second Event")),
    UTI = factor(if_else(UTI == "With", "With UTI", "Without UTI"),
                 levels = c("Without UTI", "With UTI")),
    Comparison = factor(Comparison, levels = recurrence_labels),
    risk_label = case_when(
      high_risk_with_time == "high_risk" ~ "High Risk",
      high_risk_with_time == "low_risk"  ~ "Low Risk"
    )
  )

# High-risk recurrence rate
lr_high <- likelihood_df %>% filter(risk_label == "High Risk")
median_hr <- median(lr_high$pct_recurred, na.rm = TRUE)

high_risk_likelihood <- ggplot(lr_high,
       aes(x = Time_Cutoff_label, y = Starting_Point, fill = pct_recurred)) +
  geom_tile(color = "white", linewidth = 0.5) +
  geom_text(aes(label = sprintf("%.1f%%", pct_recurred)), size = 2.8) +
  facet_grid(UTI ~ Comparison) +
  scale_fill_gradient2(low = "#1a9850", mid = "yellow", high = "red3",
                       midpoint = median_hr, name = "% Recurred") +
  labs(x = "Threshold for 'Short Time' Definition",
       y = "Starting Point",
       title = "Likelihood of Recurrence: High-Risk Group",
       subtitle = "            Subsequent Symptomatic Events needed for 'Recurrence' Definition") +
  theme_bw(base_size = 11) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        strip.text = element_text(face = "bold"), legend.position = "right")
high_risk_likelihood

# Low-risk recurrence rate
lr_low <- likelihood_df %>% filter(risk_label == "Low Risk")
median_lr <- median(lr_low$pct_recurred, na.rm = TRUE)

low_risk_likelihood <- ggplot(lr_low,
       aes(x = Time_Cutoff_label, y = Starting_Point, fill = pct_recurred)) +
  geom_tile(color = "white", linewidth = 0.5) +
  geom_text(aes(label = sprintf("%.1f%%", pct_recurred)), size = 2.8) +
  facet_grid(UTI ~ Comparison) +
  scale_fill_gradient2(low = "#1a9850", mid = "yellow", high = "red3",
                       midpoint = median_lr, name = "% Recurred") +
  labs(x = "Threshold for 'Short Time' Definition",
       y = "Starting Point",
       title = "Likelihood of Recurrence: Low-Risk Group",
       subtitle = "            Subsequent Symptomatic Events needed for 'Recurrence' Definition") +
  theme_bw(base_size = 11) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        strip.text = element_text(face = "bold"), legend.position = "right")
low_risk_likelihood

### Combined Heatmap

In [ ]:
%%R
auc_heatmap / sens_heatmap / spec_heatmap / ppv_heatmap / npv_heatmap + plot_layout(axes = "collect")

## Save Outputs

In [ ]:
%%R

# Build comprehensive results CSV with 95% CIs and likelihood of recurrence
csv_output <- all_metrics_combined %>%
  dplyr::select(
    Starting_Point, Time_Cutoff_Months, UTI,
    Recurrence_Definition = Comparison,
    N, N_High_Risk, N_Low_Risk,
    AUC_val, AUC_lo, AUC_hi,
    Sensitivity_val, Sensitivity_lo, Sensitivity_hi,
    Specificity_val, Specificity_lo, Specificity_hi,
    PPV_val, PPV_lo, PPV_hi,
    NPV_val, NPV_lo, NPV_hi,
    Likelihood_Recurrence_High_Risk_pct,
    Likelihood_Recurrence_High_Risk_lo,
    Likelihood_Recurrence_High_Risk_hi,
    N_Recurred_High_Risk,
    Likelihood_Recurrence_Low_Risk_pct,
    Likelihood_Recurrence_Low_Risk_lo,
    Likelihood_Recurrence_Low_Risk_hi,
    N_Recurred_Low_Risk
  ) %>%
  mutate(
    AUC_95CI = sprintf("%.3f (%.3f-%.3f)", AUC_val, AUC_lo, AUC_hi),
    Sensitivity_95CI = sprintf("%.3f (%.3f-%.3f)", Sensitivity_val, Sensitivity_lo, Sensitivity_hi),
    Specificity_95CI = sprintf("%.3f (%.3f-%.3f)", Specificity_val, Specificity_lo, Specificity_hi),
    PPV_95CI = sprintf("%.3f (%.3f-%.3f)", PPV_val, PPV_lo, PPV_hi),
    NPV_95CI = sprintf("%.3f (%.3f-%.3f)", NPV_val, NPV_lo, NPV_hi),
    Likelihood_High_Risk_95CI = sprintf("%.1f%% (%.1f-%.1f%%)",
      Likelihood_Recurrence_High_Risk_pct,
      Likelihood_Recurrence_High_Risk_lo,
      Likelihood_Recurrence_High_Risk_hi),
    Likelihood_Low_Risk_95CI = sprintf("%.1f%% (%.1f-%.1f%%)",
      Likelihood_Recurrence_Low_Risk_pct,
      Likelihood_Recurrence_Low_Risk_lo,
      Likelihood_Recurrence_Low_Risk_hi)
  )

write.csv(csv_output, "prognostic_accuracy_grid_search_results_aou.csv", row.names = FALSE)
cat("CSV saved with", nrow(csv_output), "rows\n")

csv_output %>%
  dplyr::select(Starting_Point, Time_Cutoff_Months, UTI, Recurrence_Definition,
                N, N_High_Risk, N_Low_Risk,
                AUC_95CI, Sensitivity_95CI, Specificity_95CI,
                PPV_95CI, NPV_95CI,
                Likelihood_High_Risk_95CI, Likelihood_Low_Risk_95CI)

# Save heatmaps
if (nrow(heatmap_data) > 0) {
  ggsave("grid_search_heatmap_auc_aou.pdf", plot = auc_heatmap, width = 12, height = 8)
  ggsave("grid_search_heatmap_sensitivity_aou.pdf", plot = sens_heatmap, width = 12, height = 8)
  ggsave("grid_search_heatmap_specificity_aou.pdf", plot = spec_heatmap, width = 12, height = 8)
  ggsave("grid_search_heatmap_ppv_aou.pdf", plot = ppv_heatmap, width = 12, height = 8)
  ggsave("grid_search_heatmap_npv_aou.pdf", plot = npv_heatmap, width = 12, height = 8)
  ggsave("grid_search_heatmap_likelihood_high_risk_aou.pdf", plot = high_risk_likelihood, width = 12, height = 10)
  ggsave("grid_search_heatmap_likelihood_low_risk_aou.pdf", plot = low_risk_likelihood, width = 12, height = 10)
}

# Save individual plots
if (!is.null(roc_index)) ggsave("grid_search_roc_index_date_no_uti_aou.pdf", plot = roc_index, width = 8, height = 6)
if (!is.null(dca_index)) ggsave("grid_search_dca_index_date_no_uti_aou.pdf", plot = dca_index, width = 10, height = 6)
if (!is.null(km_index)) { pdf("grid_search_km_index_date_no_uti_aou.pdf", width = 10, height = 7); print(km_index); dev.off() }
if (!is.null(roc_index_uti)) ggsave("grid_search_roc_index_date_with_uti_aou.pdf", plot = roc_index_uti, width = 8, height = 6)
if (!is.null(dca_index_uti)) ggsave("grid_search_dca_index_date_with_uti_aou.pdf", plot = dca_index_uti, width = 10, height = 6)
if (!is.null(km_index_uti)) { pdf("grid_search_km_index_date_with_uti_aou.pdf", width = 10, height = 7); print(km_index_uti); dev.off() }

save_grid_plots <- function(results, prefix) {
  for (nm in names(results$roc_plots)) {
    ggsave(paste0("grid_search_roc_", prefix, "_", nm, "_aou.pdf"),
           plot = results$roc_plots[[nm]], width = 8, height = 6)
  }
  for (nm in names(results$km_plots)) {
    pdf(paste0("grid_search_km_", prefix, "_", nm, "_aou.pdf"), width = 10, height = 7)
    print(results$km_plots[[nm]])
    dev.off()
  }
  for (nm in names(results$dca_plots)) {
    ggsave(paste0("grid_search_dca_", prefix, "_", nm, "_aou.pdf"),
           plot = results$dca_plots[[nm]], width = 10, height = 6)
  }
}

save_grid_plots(results_first_no_uti, "second_event_no_uti")
save_grid_plots(results_first_with_uti, "second_event_with_uti")
save_grid_plots(results_second_no_uti, "third_event_no_uti")
save_grid_plots(results_second_with_uti, "third_event_with_uti")
save_grid_plots(results_third_no_uti, "fourth_event_no_uti")
save_grid_plots(results_third_with_uti, "fourth_event_with_uti")

cat("All outputs saved.\n")

# Recurrent Disease Definition

This section investigates whether **number of previous stone events** (recurrent disease) is itself a useful risk classifier, and whether combining it with short inter-event time improves prognostication.

## Long-Format Transition Tibble

In [ ]:
%%R

transition_specs <- tribble(
  ~transition, ~from_col,     ~to_col,      ~from_event_num, ~gap_col,
  "1\u21922", "index_date",  "first_date",  1L, NA_character_,
  "2\u21923", "first_date",  "second_date", 2L, "gap_index_to_first_months",
  "3\u21924", "second_date", "third_date",  3L, "gap_first_to_second_months",
  "4\u21925", "third_date",  "fourth_date", 4L, "gap_second_to_third_months"
)

# Add gap columns not already present
if (!"gap_second_to_third_months" %in% names(dates_detailed)) {
  dates_detailed <- dates_detailed %>%
    mutate(gap_second_to_third_months = as.numeric(third_date - second_date) / 30.44)
}
if (!"gap_third_to_fourth_months" %in% names(dates_detailed)) {
  dates_detailed <- dates_detailed %>%
    mutate(gap_third_to_fourth_months = as.numeric(fourth_date - third_date) / 30.44)
}

last_date_df <- overall_follow_up %>% dplyr::select(participant_id, last_date)

transitions_long <- pmap_dfr(transition_specs, function(transition, from_col, to_col,
                                                          from_event_num, gap_col) {
  df <- dates_detailed %>%
    dplyr::select(participant_id,
                  from_date = !!sym(from_col),
                  to_date   = !!sym(to_col)) %>%
    filter(!is.na(from_date)) %>%
    left_join(last_date_df, by = "participant_id") %>%
    filter(!is.na(last_date)) %>%
    mutate(
      transition       = transition,
      from_event_num   = from_event_num,
      had_next_event   = if_else(!is.na(to_date), 1L, 0L),
      event_or_censor_date = if_else(!is.na(to_date), to_date, last_date),
      time_to_next_years   = as.numeric(event_or_censor_date - from_date) / 365.25,
      time_to_next_years   = pmin(time_to_next_years, follow_up_years),
      had_next_event   = if_else(
        !is.na(to_date) & as.numeric(to_date - from_date) / 365.25 > follow_up_years,
        0L, had_next_event),
      n_previous_events = from_event_num
    )

  if (!is.na(gap_col)) {
    gap_vals <- dates_detailed %>%
      dplyr::select(participant_id, prior_gap_months = !!sym(gap_col))
    df <- df %>% left_join(gap_vals, by = "participant_id")
  } else {
    df <- df %>% mutate(prior_gap_months = NA_real_)
  }

  df %>%
    dplyr::select(participant_id, transition, from_event_num, from_date,
                  to_date, last_date, time_to_next_years, had_next_event,
                  n_previous_events, prior_gap_months) %>%
    filter(time_to_next_years > 0)
})

transitions_long$transition <- factor(transitions_long$transition,
  levels = c("1\u21922", "2\u21923", "3\u21924", "4\u21925"))

cat("Transitions summary (AoU):\n")
transitions_long %>%
  group_by(transition) %>%
  summarise(n=n(), n_events=sum(had_next_event),
            pct_event=round(100*mean(had_next_event),1), .groups="drop") %>%
  print()


## KM by Event Number (Transition Type)

In [ ]:
%%R

fit_transition <- surv_fit(Surv(time_to_next_years, had_next_event) ~ transition,
                           data = transitions_long)

km_by_event <- ggsurvplot(
  fit_transition, data = transitions_long,
  pval = TRUE, pval.method = TRUE,
  conf.int = TRUE, censor = FALSE,
  risk.table = TRUE, risk.table.height = 0.3,
  surv.median.line = "hv", break.time.by = 1,
  xlab = "Time (years)", ylab = "Event-free probability",
  title = "Time to Next Stone Event by Transition (AoU)",
  legend.title = "Transition",
  palette = c("#1b9e77","#d95f02","#7570b3","#e7298a"),
  xlim = c(0, follow_up_years)
)
print(km_by_event)
ggsave("recurrent_disease_km_by_transition_aou.pdf",
       plot = km_by_event$plot, width = 10, height = 7)


### Faceted by Transition, Stratified by Total Event Count

In [ ]:
%%R

total_events <- dates_detailed %>%
  dplyr::select(participant_id, n_recurrences) %>%
  mutate(total_events = n_recurrences + 1L)

transitions_long <- transitions_long %>%
  left_join(total_events, by = "participant_id") %>%
  mutate(
    total_events_grp = case_when(
      total_events == 1 ~ "1 event",
      total_events == 2 ~ "2 events",
      total_events == 3 ~ "3 events",
      total_events == 4 ~ "4 events",
      total_events >= 5 ~ ">4 events"
    ),
    total_events_grp = factor(total_events_grp,
      levels = c("1 event","2 events","3 events","4 events",">4 events"))
  )

transition_levels <- levels(transitions_long$transition)
facet_km_plots <- lapply(transition_levels, function(tr) {
  tr_data <- transitions_long %>% filter(transition == tr)
  grp_counts <- tr_data %>% count(total_events_grp) %>% filter(n >= 5)
  tr_data <- tr_data %>%
    filter(total_events_grp %in% grp_counts$total_events_grp) %>% droplevels()
  if (nlevels(tr_data$total_events_grp) < 2) return(NULL)
  fit <- surv_fit(Surv(time_to_next_years, had_next_event) ~ total_events_grp, data=tr_data)
  ggsurvplot(fit, data=tr_data,
    pval=TRUE, pval.method=TRUE, conf.int=TRUE, censor=FALSE,
    risk.table=TRUE, risk.table.height=0.35, surv.median.line="hv",
    xlab="Time (years)", ylab="Event-free probability",
    title=paste0("Transition ", tr, ": Stratified by Total Events (AoU)"),
    legend.title="Total Events", palette="jco",
    xlim=c(0,follow_up_years), break.time.by=1)
})
for (p in facet_km_plots) if (!is.null(p)) print(p)
for (i in seq_along(transition_levels)) {
  if (!is.null(facet_km_plots[[i]])) {
    fname <- paste0("recurrent_disease_km_transition_",
                    gsub("\u2192","to",transition_levels[i]),
                    "_by_total_events_aou.pdf")
    ggsave(fname, plot=facet_km_plots[[i]]$plot, width=10, height=7)
  }
}


## KM by Recurrent Disease (Event Count)

In [ ]:
%%R

transitions_long <- transitions_long %>%
  mutate(
    recurrent_2plus = if_else(from_event_num >= 2,
                              "High Risk (\u22652 events)", "Low Risk (1 event)"),
    recurrent_3plus = if_else(from_event_num >= 3,
                              "High Risk (\u22653 events)", "Low Risk (<3 events)")
  )

fit_2plus <- surv_fit(Surv(time_to_next_years, had_next_event) ~ recurrent_2plus,
                      data = transitions_long)
km_2plus <- ggsurvplot(fit_2plus, data=transitions_long,
  pval=TRUE, pval.method=TRUE, conf.int=TRUE, censor=FALSE,
  risk.table=TRUE, risk.table.height=0.25, surv.median.line="hv",
  xlab="Time (years)", ylab="Event-free probability",
  title="Recurrent Disease: \u22652 Previous Events vs 1 Event (AoU)",
  legend.title="Risk Group", palette=c("#d95f02","#1b9e77"),
  xlim=c(0,follow_up_years), break.time.by=1)
print(km_2plus)
ggsave("recurrent_disease_km_2plus_aou.pdf", plot=km_2plus$plot, width=10, height=7)

fit_3plus <- surv_fit(Surv(time_to_next_years, had_next_event) ~ recurrent_3plus,
                      data = transitions_long)
km_3plus <- ggsurvplot(fit_3plus, data=transitions_long,
  pval=TRUE, pval.method=TRUE, conf.int=TRUE, censor=FALSE,
  risk.table=TRUE, risk.table.height=0.25, surv.median.line="hv",
  xlab="Time (years)", ylab="Event-free probability",
  title="Recurrent Disease: \u22653 Previous Events vs <3 Events (AoU)",
  legend.title="Risk Group", palette=c("#d95f02","#1b9e77"),
  xlim=c(0,follow_up_years), break.time.by=1)
print(km_3plus)
ggsave("recurrent_disease_km_3plus_aou.pdf", plot=km_3plus$plot, width=10, height=7)


## Short Inter-Event Time KMs and Three-Group Comparison

In [ ]:
%%R

transitions_with_gap <- transitions_long %>%
  filter(from_event_num >= 2, !is.na(prior_gap_months))

for (cutoff in c(12, 18, 24)) {
  transitions_with_gap <- transitions_with_gap %>%
    mutate(!!paste0("risk_short_",cutoff) := case_when(
      prior_gap_months <= cutoff ~ paste0("High Risk (recurrent + gap \u2264",cutoff,"mo)"),
      TRUE ~ paste0("Recurrent (gap >",cutoff,"mo)")))

  risk_col <- paste0("risk_short_",cutoff)
  fit_s <- surv_fit(as.formula(paste0("Surv(time_to_next_years,had_next_event)~",risk_col)),
                    data=transitions_with_gap)
  km_s <- ggsurvplot(fit_s, data=transitions_with_gap,
    pval=TRUE, pval.method=TRUE, conf.int=TRUE, censor=FALSE,
    risk.table=TRUE, risk.table.height=0.25, surv.median.line="hv",
    xlab="Time (years)", ylab="Event-free probability",
    title=paste0("Recurrent Patients: Short Gap (\u2264",cutoff,"mo) vs Long Gap (AoU)"),
    legend.title="Risk Group", palette=c("#d95f02","#1b9e77"),
    xlim=c(0,follow_up_years), break.time.by=1)
  print(km_s)
  ggsave(paste0("recurrent_disease_km_short_gap_",cutoff,"mo_aou.pdf"),
         plot=km_s$plot, width=10, height=7)
}

for (cutoff in c(12, 18, 24)) {
  transitions_long <- transitions_long %>%
    mutate(!!paste0("three_group_",cutoff) := case_when(
      from_event_num == 1 ~ "First Event",
      from_event_num >= 2 & !is.na(prior_gap_months) & prior_gap_months <= cutoff ~
        paste0("Recurrent + Short Gap (\u2264",cutoff,"mo)"),
      from_event_num >= 2 ~ paste0("Recurrent + Long Gap (>",cutoff,"mo)")))

  risk_col <- paste0("three_group_",cutoff)
  fit_3g <- surv_fit(as.formula(paste0("Surv(time_to_next_years,had_next_event)~",risk_col)),
                     data=transitions_long)
  km_3g <- ggsurvplot(fit_3g, data=transitions_long,
    pval=TRUE, pval.method=TRUE, conf.int=TRUE, censor=FALSE,
    risk.table=TRUE, risk.table.height=0.3, surv.median.line="hv",
    xlab="Time (years)", ylab="Event-free probability",
    title=paste0("Three-Group Risk: First vs Recurrent Gap \u2264/>",cutoff,"mo (AoU)"),
    legend.title="Risk Group", palette=c("#1b9e77","#d95f02","#7570b3"),
    xlim=c(0,follow_up_years), break.time.by=1)
  print(km_3g)
  ggsave(paste0("recurrent_disease_km_three_group_",cutoff,"mo_aou.pdf"),
         plot=km_3g$plot, width=10, height=7)
}


## Combined EAU + Recurrent Disease (AoU)

In [ ]:
%%R

eau_risk_by_start <- bind_rows(
  build_high_risk_df("index_date",  include_uti=FALSE) %>% mutate(from_col="index_date",  from_event_num=1L),
  build_high_risk_df("first_date",  include_uti=FALSE) %>% mutate(from_col="first_date",  from_event_num=2L),
  build_high_risk_df("second_date", include_uti=FALSE) %>% mutate(from_col="second_date", from_event_num=3L),
  build_high_risk_df("third_date",  include_uti=FALSE) %>% mutate(from_col="third_date",  from_event_num=4L)
)

transitions_long <- transitions_long %>%
  left_join(eau_risk_by_start %>% dplyr::select(participant_id, from_event_num, eau_risk=high_risk),
            by = c("participant_id","from_event_num")) %>%
  mutate(
    recurrent_short_12 = from_event_num >= 2 & !is.na(prior_gap_months) & prior_gap_months <= 12,
    risk_eau_only      = if_else(!is.na(eau_risk) & eau_risk=="high_risk","EAU High Risk","Low Risk"),
    risk_combined      = case_when(
      eau_risk=="high_risk" | recurrent_short_12 ~ "High Risk (EAU or Recurrent+Short)",
      TRUE ~ "Low Risk"),
    risk_four_group    = case_when(
      eau_risk=="high_risk" & recurrent_short_12 ~ "EAU High + Recurrent Short Gap",
      eau_risk=="high_risk" & !recurrent_short_12 ~ "EAU High Only",
      (!is.na(eau_risk) & eau_risk!="high_risk") & recurrent_short_12 ~ "Recurrent Short Gap Only",
      TRUE ~ "Low Risk")
  )

fit_eau <- surv_fit(Surv(time_to_next_years,had_next_event)~risk_eau_only, data=transitions_long)
km_eau <- ggsurvplot(fit_eau, data=transitions_long,
  pval=TRUE, pval.method=TRUE, conf.int=TRUE, censor=FALSE,
  risk.table=TRUE, risk.table.height=0.25, surv.median.line="hv",
  xlab="Time (years)", ylab="Event-free probability",
  title="EAU Risk Classification Only \u2013 Pooled Transitions (AoU)",
  legend.title="Risk Group", palette=c("#d95f02","#1b9e77"),
  xlim=c(0,follow_up_years), break.time.by=1)
print(km_eau)

fit_combined <- surv_fit(Surv(time_to_next_years,had_next_event)~risk_combined, data=transitions_long)
km_combined <- ggsurvplot(fit_combined, data=transitions_long,
  pval=TRUE, pval.method=TRUE, conf.int=TRUE, censor=FALSE,
  risk.table=TRUE, risk.table.height=0.25, surv.median.line="hv",
  xlab="Time (years)", ylab="Event-free probability",
  title="Combined Risk: EAU OR Recurrent Disease (\u22652 events + gap \u226412mo) (AoU)",
  legend.title="Risk Group", palette=c("#d95f02","#1b9e77"),
  xlim=c(0,follow_up_years), break.time.by=1)
print(km_combined)

fit_four <- surv_fit(Surv(time_to_next_years,had_next_event)~risk_four_group, data=transitions_long)
km_four <- ggsurvplot(fit_four, data=transitions_long,
  pval=TRUE, pval.method=TRUE, conf.int=TRUE, censor=FALSE,
  risk.table=TRUE, risk.table.height=0.35, surv.median.line="hv",
  xlab="Time (years)", ylab="Event-free probability",
  title="Four-Group Risk Breakdown: EAU \u00d7 Recurrent Disease 12mo (AoU)",
  legend.title="Risk Group", palette=c("#e7298a","#d95f02","#7570b3","#1b9e77"),
  xlim=c(0,follow_up_years), break.time.by=1)
print(km_four)

ggsave("recurrent_disease_km_eau_only_pooled_aou.pdf",       plot=km_eau$plot,      width=10, height=7)
ggsave("recurrent_disease_km_combined_eau_recurrent_aou.pdf",plot=km_combined$plot, width=10, height=7)
ggsave("recurrent_disease_km_four_group_aou.pdf",            plot=km_four$plot,     width=12, height=8)


## Prognostic Accuracy (Recurrent Disease Combined Classifier) – AoU

In [ ]:
%%R

rec_metrics_list_aou <- list()
rec_roc_plots_aou    <- list()
rec_km_plots_aou     <- list()
rec_dca_plots_aou    <- list()
rec_label <- "\u22651 subsequent stone event"

for (cutoff in time_cutoffs_months) {
  for (tr in levels(transitions_long$transition)) {
    tl <- transitions_long %>%
      filter(transition == tr) %>%
      mutate(
        recurrent_short   = from_event_num >= 2 & !is.na(prior_gap_months) & prior_gap_months <= cutoff,
        risk_combined_cut = if_else(!is.na(eau_risk) & (eau_risk=="high_risk" | recurrent_short),
                                    "high_risk","low_risk"),
        risk_eau          = if_else(!is.na(eau_risk) & eau_risk=="high_risk","high_risk","low_risk"),
        subsequent_events = had_next_event,
        follow_up_from_start_years = time_to_next_years,
        has_next_event    = had_next_event
      )

    for (risk_def in c("EAU Only","EAU + Recurrent Disease")) {
      risk_col <- if (risk_def=="EAU Only") "risk_eau" else "risk_combined_cut"
      roc_data <- tl %>%
        mutate(
          actual    = case_when(
            had_next_event==1 ~ 1L,
            had_next_event==0 & time_to_next_years>=follow_up_years ~ 0L,
            TRUE ~ NA_integer_),
          predicted = if_else(!!sym(risk_col)=="high_risk",1,0)
        ) %>% drop_na(actual,predicted)
      if (nrow(roc_data)<10 || length(unique(roc_data$actual))<2) next
      roc_obj <- roc(roc_data$actual, roc_data$predicted, quiet=TRUE)
      m <- extract_metrics_grid(roc_obj, rec_label, roc_data) %>%
        mutate(Transition=tr, Starting_Point=tr,
               Time_Cutoff_Months=cutoff, UTI="Without", Risk_Definition=risk_def,
               Comparison=rec_label)
      rec_metrics_list_aou[[paste(tr,cutoff,risk_def,sep="_")]] <- m
    }

    if (cutoff==12) {
      tl12 <- tl
      # ROC overlay
      roc_ov <- data.frame()
      for (risk_def in c("EAU Only","EAU + Recurrent Disease")) {
        risk_col <- if (risk_def=="EAU Only") "risk_eau" else "risk_combined_cut"
        rd <- tl12 %>%
          mutate(actual=case_when(had_next_event==1~1L,
                                  had_next_event==0&time_to_next_years>=follow_up_years~0L,
                                  TRUE~NA_integer_),
                 predicted=if_else(!!sym(risk_col)=="high_risk",1,0)) %>%
          drop_na(actual,predicted)
        if (nrow(rd)<10||length(unique(rd$actual))<2) next
        ro <- roc(rd$actual,rd$predicted,quiet=TRUE)
        auc_v <- round(as.numeric(auc(ro)),3)
        roc_ov <- rbind(roc_ov, data.frame(fpr=1-ro$specificities, tpr=ro$sensitivities,
                                            Definition=paste0(risk_def," (AUC=",auc_v,")")))
      }
      if (nrow(roc_ov)>0) {
        p_roc <- ggplot(roc_ov, aes(x=fpr,y=tpr,colour=Definition)) +
          geom_line(linewidth=1) + geom_abline(linetype="dashed",alpha=0.5) +
          coord_equal() +
          labs(x="FPR",y="TPR",
               title=paste0("ROC: Transition ",tr," (12mo) (AoU)"),colour="Definition") +
          theme_bw(base_size=12) + theme(legend.position="bottom")
        safe_tr <- gsub("\u2192","to",tr)
        rec_roc_plots_aou[[paste0("transition_",safe_tr,"_cutoff_12")]] <- p_roc
      }
      # KM
      tl_km <- tl12 %>% rename(risk_group=risk_combined_cut) %>%
        filter(!is.na(time_to_next_years),!is.na(risk_group))
      if (nrow(tl_km)>=10 && length(unique(tl_km$risk_group))>=2) {
        km_fit <- surv_fit(Surv(time_to_next_years,had_next_event)~risk_group, data=tl_km)
        km_p <- ggsurvplot(km_fit, data=tl_km,
          pval=TRUE, pval.method=TRUE, conf.int=TRUE, censor=FALSE,
          risk.table=TRUE, risk.table.height=0.28, surv.median.line="hv", break.time.by=1,
          xlab="Time (years)", xlim=c(0,follow_up_years),
          title=paste0("KM: Transition ",tr," – EAU + Recurrent Disease (12mo) (AoU)"),
          legend.title="Risk Status", legend.labs=c("High Risk","Low Risk"),
          palette=c("#d95f02","#1b9e77"), ggtheme=theme_bw())
        safe_tr <- gsub("\u2192","to",tr)
        rec_km_plots_aou[[paste0("transition_",safe_tr,"_cutoff_12")]] <- km_p
      }
      # DCA
      tl_dca <- tl12 %>%
        mutate(actual=as.integer(had_next_event==1),
               risk_binary=if_else(risk_combined_cut=="high_risk",1,0)) %>%
        drop_na(actual,risk_binary)
      if (nrow(tl_dca)>=10 && length(unique(tl_dca$actual))>=2) {
        dca_res <- dca(actual~risk_binary, data=tl_dca, thresholds=seq(0,0.8,0.01))
        dca_df  <- plot(dca_res)$data
        p_dca <- ggplot(dca_df, aes(x=threshold,y=net_benefit,colour=label)) +
          geom_line(linewidth=0.8) +
          labs(x="Probability Threshold",y="Net Benefit",colour=NULL,
               title=paste0("DCA: Transition ",tr," (12mo) (AoU)")) +
          theme_bw(base_size=12) + theme(legend.position="bottom") +
          xlim(c(0,0.6)) + geom_vline(xintercept=0.2,linetype="dashed")
        safe_tr <- gsub("\u2192","to",tr)
        rec_dca_plots_aou[[paste0("transition_",safe_tr,"_cutoff_12")]] <- p_dca
      }
    }
  }
}

rec_metrics_df_aou <- bind_rows(rec_metrics_list_aou) %>%
  mutate(
    Time_Cutoff_label = paste0("\u2264",Time_Cutoff_Months,"mo"),
    Time_Cutoff_label = factor(Time_Cutoff_label,
      levels=c("\u22646mo","\u22649mo","\u226412mo","\u226415mo","\u226418mo","\u226421mo","\u226424mo")),
    Transition = factor(Transition, levels=c("1\u21922","2\u21923","3\u21924","4\u21925")),
    Risk_Definition = factor(Risk_Definition, levels=c("EAU Only","EAU + Recurrent Disease"))
  )

cat("AoU recurrent disease PA metrics:", nrow(rec_metrics_df_aou), "rows\n")

# Display plots
for (nm in names(rec_roc_plots_aou)) print(rec_roc_plots_aou[[nm]])
for (nm in names(rec_km_plots_aou)) print(rec_km_plots_aou[[nm]])
for (nm in names(rec_dca_plots_aou)) print(rec_dca_plots_aou[[nm]])


## Updated Heatmaps (AoU)

In [ ]:
%%R

rec_hm_aou <- rec_metrics_df_aou %>%
  mutate(Transition = factor(Transition, levels=rev(levels(Transition))))

make_rec_heatmap <- function(df, fill_col, title_str, legend_name) {
  med_val <- median(df[[fill_col]], na.rm=TRUE)
  ggplot(df, aes_string(x="Time_Cutoff_label", y="Transition", fill=fill_col)) +
    geom_tile(color="white", linewidth=0.5) +
    geom_text(aes_string(label=sprintf('sprintf("%.2f", %s)', fill_col)), size=3) +
    facet_wrap(~Risk_Definition, nrow=1) +
    scale_fill_gradient2(low="red3", mid="yellow", high="#1a9850",
                         midpoint=med_val, name=legend_name) +
    labs(x="Gap Threshold", y="Transition", title=title_str) +
    theme_bw(base_size=11) +
    theme(axis.text.x=element_text(angle=45,hjust=1),
          strip.text=element_text(face="bold"), legend.position="right")
}

rec_auc_heatmap_aou  <- make_rec_heatmap(rec_hm_aou,"AUC_val",
  "AUC: EAU Only vs EAU + Recurrent Disease (AoU)","AUC")
rec_sens_heatmap_aou <- make_rec_heatmap(rec_hm_aou,"Sensitivity_val",
  "Sensitivity (AoU)","Sensitivity")
rec_spec_heatmap_aou <- make_rec_heatmap(rec_hm_aou,"Specificity_val",
  "Specificity (AoU)","Specificity")
rec_ppv_heatmap_aou  <- make_rec_heatmap(rec_hm_aou,"PPV_val","PPV (AoU)","PPV")
rec_npv_heatmap_aou  <- make_rec_heatmap(rec_hm_aou,"NPV_val","NPV (AoU)","NPV")

print(rec_auc_heatmap_aou)
print(rec_sens_heatmap_aou)
print(rec_spec_heatmap_aou)
print(rec_ppv_heatmap_aou)
print(rec_npv_heatmap_aou)

ggsave("recurrent_disease_heatmap_auc_aou.pdf",         plot=rec_auc_heatmap_aou,  width=13,height=7)
ggsave("recurrent_disease_heatmap_sensitivity_aou.pdf", plot=rec_sens_heatmap_aou, width=13,height=7)
ggsave("recurrent_disease_heatmap_specificity_aou.pdf", plot=rec_spec_heatmap_aou, width=13,height=7)
ggsave("recurrent_disease_heatmap_ppv_aou.pdf",         plot=rec_ppv_heatmap_aou,  width=13,height=7)
ggsave("recurrent_disease_heatmap_npv_aou.pdf",         plot=rec_npv_heatmap_aou,  width=13,height=7)

for (nm in names(rec_roc_plots_aou))
  ggsave(paste0("recurrent_disease_roc_",nm,"_aou.pdf"), plot=rec_roc_plots_aou[[nm]], width=8,height=6)
for (nm in names(rec_km_plots_aou)) {
  pdf(paste0("recurrent_disease_km_prognostic_",nm,"_aou.pdf"), width=10, height=7)
  print(rec_km_plots_aou[[nm]]); dev.off()
}
for (nm in names(rec_dca_plots_aou))
  ggsave(paste0("recurrent_disease_dca_",nm,"_aou.pdf"), plot=rec_dca_plots_aou[[nm]], width=10,height=6)

write.csv(rec_metrics_df_aou %>%
  dplyr::select(Transition,Starting_Point,Time_Cutoff_Months,Risk_Definition,
                N,N_High_Risk,N_Low_Risk,AUC_val,AUC_lo,AUC_hi,
                Sensitivity_val,Sensitivity_lo,Sensitivity_hi,
                Specificity_val,Specificity_lo,Specificity_hi,
                PPV_val,PPV_lo,PPV_hi,NPV_val,NPV_lo,NPV_hi,
                Likelihood_Recurrence_High_Risk_pct,Likelihood_Recurrence_High_Risk_lo,
                Likelihood_Recurrence_High_Risk_hi,
                Likelihood_Recurrence_Low_Risk_pct,Likelihood_Recurrence_Low_Risk_lo,
                Likelihood_Recurrence_Low_Risk_hi),
  "recurrent_disease_prognostic_accuracy_results_aou.csv", row.names=FALSE)
cat("All AoU recurrent disease outputs saved.\n")


# EAU vs EAU + Prior Event Count (No Time Cutoff) – AoU

Replicates the UK Biobank analysis for AoU. Sweeps count thresholds (≥2, ≥3, ≥4 prior events) as additions to EAU risk, with no time-gap requirement. Two analyses: per-transition and pooled (one row per participant using latest observed transition).


In [ ]:
%%R

count_thresholds <- c(2L, 3L, 4L)

pertr_metrics_list_aou <- list()
pertr_roc_plots_aou    <- list()
pertr_delong_rows_aou  <- list()

for (tr in levels(transitions_long$transition)) {

  tl <- transitions_long %>%
    filter(transition == tr) %>%
    mutate(risk_eau = if_else(!is.na(eau_risk) & eau_risk == "high_risk",
                              "high_risk", "low_risk"))

  rd_eau <- tl %>%
    mutate(
      actual = case_when(
        had_next_event == 1 ~ 1L,
        had_next_event == 0 & time_to_next_years >= follow_up_years ~ 0L,
        TRUE ~ NA_integer_
      ),
      predicted = if_else(risk_eau == "high_risk", 1, 0)
    ) %>% drop_na(actual, predicted)

  if (nrow(rd_eau) < 10 || length(unique(rd_eau$actual)) < 2) next

  ro_eau <- roc(rd_eau$actual, rd_eau$predicted, quiet = TRUE)
  auc_eau <- round(as.numeric(auc(ro_eau)), 3)

  m_eau <- extract_metrics_grid(ro_eau, "\u22651 subsequent stone event", rd_eau) %>%
    mutate(Transition = tr, Risk_Definition = "EAU Only",
           Count_Threshold = NA_integer_)
  pertr_metrics_list_aou[[paste(tr, "EAU_Only", sep = "_")]] <- m_eau

  roc_overlay_df <- data.frame(
    fpr = 1 - ro_eau$specificities,
    tpr = ro_eau$sensitivities,
    Definition = paste0("EAU Only (AUC=", auc_eau, ")")
  )

  for (ct in count_thresholds) {
    tl2 <- tl %>%
      mutate(
        recurrent_count = from_event_num >= ct,
        risk_combined = if_else(
          !is.na(eau_risk) & (eau_risk == "high_risk" | recurrent_count),
          "high_risk", "low_risk"
        )
      )

    rd_c <- tl2 %>%
      mutate(
        actual = case_when(
          had_next_event == 1 ~ 1L,
          had_next_event == 0 & time_to_next_years >= follow_up_years ~ 0L,
          TRUE ~ NA_integer_
        ),
        predicted = if_else(risk_combined == "high_risk", 1, 0)
      ) %>% drop_na(actual, predicted)

    if (nrow(rd_c) < 10 || length(unique(rd_c$actual)) < 2) next

    ro_c <- roc(rd_c$actual, rd_c$predicted, quiet = TRUE)
    auc_c <- round(as.numeric(auc(ro_c)), 3)

    m_c <- extract_metrics_grid(ro_c, "\u22651 subsequent stone event", rd_c) %>%
      mutate(Transition = tr,
             Risk_Definition = paste0("EAU + \u2265", ct, " events"),
             Count_Threshold = ct)
    pertr_metrics_list_aou[[paste(tr, "count", ct, sep = "_")]] <- m_c

    roc_overlay_df <- rbind(roc_overlay_df, data.frame(
      fpr = 1 - ro_c$specificities,
      tpr = ro_c$sensitivities,
      Definition = paste0("EAU + \u2265", ct, " events (AUC=", auc_c, ")")
    ))

    dl <- tryCatch(roc.test(ro_eau, ro_c, method = "delong"),
                   error = function(e) NULL)
    pertr_delong_rows_aou[[paste(tr, ct, sep = "_")]] <- data.frame(
      Transition = tr, Count_Threshold = ct,
      Risk_Definition = paste0("EAU + \u2265", ct, " events"),
      delong_p = if (is.null(dl)) NA_real_ else as.numeric(dl$p.value)
    )
  }

  p_roc <- ggplot(roc_overlay_df, aes(x = fpr, y = tpr, colour = Definition)) +
    geom_line(linewidth = 1) +
    geom_abline(linetype = "dashed", alpha = 0.5) +
    coord_equal() +
    labs(x = "False Positive Rate", y = "True Positive Rate",
         title = paste0("ROC (AoU): Transition ", tr,
                        " \u2014 EAU vs EAU + prior event count"),
         colour = "Risk Definition") +
    scale_colour_manual(values = c("#7570b3", "#d95f02", "#1b9e77", "#e7298a")) +
    theme_bw(base_size = 12) +
    theme(legend.position = "bottom") +
    guides(colour = guide_legend(nrow = 2))

  safe_tr <- gsub("\u2192", "to", tr)
  pertr_roc_plots_aou[[paste0("transition_", safe_tr)]] <- p_roc
}

pertr_metrics_df_aou <- bind_rows(pertr_metrics_list_aou) %>%
  mutate(Transition = factor(Transition,
                             levels = c("1\u21922","2\u21923","3\u21924","4\u21925")),
         Risk_Definition = factor(Risk_Definition,
                                  levels = c("EAU Only",
                                             "EAU + \u22652 events",
                                             "EAU + \u22653 events",
                                             "EAU + \u22654 events")))

pertr_delong_df_aou <- bind_rows(pertr_delong_rows_aou) %>%
  mutate(delong_p_fmt = ifelse(is.na(delong_p), NA,
                               ifelse(delong_p < 0.001, "<0.001",
                                      sprintf("%.3f", delong_p))))

cat("AoU per-transition sweep complete:", nrow(pertr_metrics_df_aou), "rows.\n")

for (nm in names(pertr_roc_plots_aou)) {
  ggsave(paste0("eau_vs_count_no_timecutoff_roc_", nm, "_aou.pdf"),
         plot = pertr_roc_plots_aou[[nm]], width = 9, height = 7)
}

write.csv(
  pertr_metrics_df_aou %>%
    left_join(pertr_delong_df_aou %>%
                dplyr::select(Transition, Risk_Definition, delong_p),
              by = c("Transition", "Risk_Definition")) %>%
    dplyr::select(Transition, Risk_Definition, Count_Threshold,
                  N, N_High_Risk, N_Low_Risk,
                  AUC_val, AUC_lo, AUC_hi, delong_p,
                  Sensitivity_val, Sensitivity_lo, Sensitivity_hi,
                  Specificity_val, Specificity_lo, Specificity_hi,
                  PPV_val, PPV_lo, PPV_hi,
                  NPV_val, NPV_lo, NPV_hi),
  "eau_vs_count_no_timecutoff_per_transition_aou.csv",
  row.names = FALSE)

print(pertr_metrics_df_aou %>%
  left_join(pertr_delong_df_aou %>%
              dplyr::select(Transition, Risk_Definition, delong_p_fmt),
            by = c("Transition", "Risk_Definition")) %>%
  dplyr::select(Transition, Risk_Definition, N, N_High_Risk, N_Low_Risk,
                AUC, delong_p_fmt, Sensitivity, Specificity, PPV, NPV) %>%
  arrange(Transition, Risk_Definition))


## Pooled Analysis (AoU): One Row per Participant (Latest Transition)


In [ ]:
%%R

pooled_df_aou <- transitions_long %>%
  mutate(usable = case_when(
    had_next_event == 1 ~ TRUE,
    had_next_event == 0 & time_to_next_years >= follow_up_years ~ TRUE,
    TRUE ~ FALSE)) %>%
  filter(usable) %>%
  group_by(participant_id) %>%
  arrange(desc(from_event_num)) %>%
  slice(1) %>%
  ungroup() %>%
  mutate(actual = case_when(
           had_next_event == 1 ~ 1L,
           had_next_event == 0 & time_to_next_years >= follow_up_years ~ 0L,
           TRUE ~ NA_integer_),
         risk_eau = if_else(!is.na(eau_risk) & eau_risk == "high_risk",
                            "high_risk", "low_risk")) %>%
  drop_na(actual)

cat("AoU pooled N =", nrow(pooled_df_aou), "\n")
print(table(pooled_df_aou$from_event_num))

pooled_metrics_list_aou <- list()
pooled_delong_rows_aou  <- list()
pooled_roc_overlay_aou  <- data.frame()

rd_eau_p <- pooled_df_aou %>%
  mutate(predicted = if_else(risk_eau == "high_risk", 1, 0)) %>%
  drop_na(actual, predicted)
ro_eau_p <- roc(rd_eau_p$actual, rd_eau_p$predicted, quiet = TRUE)
auc_eau_p <- round(as.numeric(auc(ro_eau_p)), 3)

pooled_metrics_list_aou[["EAU_Only"]] <-
  extract_metrics_grid(ro_eau_p, "\u22651 subsequent stone event", rd_eau_p) %>%
  mutate(Risk_Definition = "EAU Only", Count_Threshold = NA_integer_)

pooled_roc_overlay_aou <- rbind(pooled_roc_overlay_aou, data.frame(
  fpr = 1 - ro_eau_p$specificities,
  tpr = ro_eau_p$sensitivities,
  Definition = paste0("EAU Only (AUC=", auc_eau_p, ")")))

for (ct in count_thresholds) {
  rd_c_p <- pooled_df_aou %>%
    mutate(recurrent_count = from_event_num >= ct,
           risk_combined = if_else(
             !is.na(eau_risk) & (eau_risk == "high_risk" | recurrent_count),
             "high_risk", "low_risk"),
           predicted = if_else(risk_combined == "high_risk", 1, 0)) %>%
    drop_na(actual, predicted)

  if (length(unique(rd_c_p$predicted)) < 2) next

  ro_c_p <- roc(rd_c_p$actual, rd_c_p$predicted, quiet = TRUE)
  auc_c_p <- round(as.numeric(auc(ro_c_p)), 3)

  pooled_metrics_list_aou[[paste0("count_", ct)]] <-
    extract_metrics_grid(ro_c_p, "\u22651 subsequent stone event", rd_c_p) %>%
    mutate(Risk_Definition = paste0("EAU + \u2265", ct, " events"),
           Count_Threshold = ct)

  pooled_roc_overlay_aou <- rbind(pooled_roc_overlay_aou, data.frame(
    fpr = 1 - ro_c_p$specificities,
    tpr = ro_c_p$sensitivities,
    Definition = paste0("EAU + \u2265", ct, " events (AUC=", auc_c_p, ")")))

  dl_p <- tryCatch(roc.test(ro_eau_p, ro_c_p, method = "delong"),
                   error = function(e) NULL)
  pooled_delong_rows_aou[[as.character(ct)]] <- data.frame(
    Count_Threshold = ct,
    Risk_Definition = paste0("EAU + \u2265", ct, " events"),
    delong_p = if (is.null(dl_p)) NA_real_ else as.numeric(dl_p$p.value))
}

pooled_metrics_df_aou <- bind_rows(pooled_metrics_list_aou) %>%
  mutate(Risk_Definition = factor(Risk_Definition,
    levels = c("EAU Only","EAU + \u22652 events",
               "EAU + \u22653 events","EAU + \u22654 events")))

pooled_delong_df_aou <- bind_rows(pooled_delong_rows_aou) %>%
  mutate(delong_p_fmt = ifelse(is.na(delong_p), NA,
                               ifelse(delong_p < 0.001, "<0.001",
                                      sprintf("%.3f", delong_p))))

pooled_roc_plot_aou <- ggplot(pooled_roc_overlay_aou,
                              aes(x = fpr, y = tpr, colour = Definition)) +
  geom_line(linewidth = 1) +
  geom_abline(linetype = "dashed", alpha = 0.5) +
  coord_equal() +
  labs(x = "False Positive Rate", y = "True Positive Rate",
       title = "Pooled ROC (AoU): EAU vs EAU + Prior Event Count (no time cutoff)",
       subtitle = paste0("N = ", nrow(pooled_df_aou),
                         " (one row per participant, latest observed transition)"),
       colour = "Risk Definition") +
  scale_colour_manual(values = c("#7570b3", "#d95f02", "#1b9e77", "#e7298a")) +
  theme_bw(base_size = 12) +
  theme(legend.position = "bottom") +
  guides(colour = guide_legend(nrow = 2))

print(pooled_roc_plot_aou)

ggsave("eau_vs_count_no_timecutoff_pooled_roc_aou.pdf",
       plot = pooled_roc_plot_aou, width = 9, height = 7)

write.csv(
  pooled_metrics_df_aou %>%
    left_join(pooled_delong_df_aou %>%
                dplyr::select(Risk_Definition, delong_p),
              by = "Risk_Definition") %>%
    dplyr::select(Risk_Definition, Count_Threshold,
                  N, N_High_Risk, N_Low_Risk,
                  AUC_val, AUC_lo, AUC_hi, delong_p,
                  Sensitivity_val, Sensitivity_lo, Sensitivity_hi,
                  Specificity_val, Specificity_lo, Specificity_hi,
                  PPV_val, PPV_lo, PPV_hi,
                  NPV_val, NPV_lo, NPV_hi),
  "eau_vs_count_no_timecutoff_pooled_aou.csv",
  row.names = FALSE)

print(pooled_metrics_df_aou %>%
  left_join(pooled_delong_df_aou %>%
              dplyr::select(Risk_Definition, delong_p_fmt),
            by = "Risk_Definition") %>%
  dplyr::select(Risk_Definition, N, N_High_Risk, N_Low_Risk,
                AUC, delong_p_fmt, Sensitivity, Specificity, PPV, NPV) %>%
  arrange(Risk_Definition))

cat("AoU EAU vs count-threshold (no time cutoff) outputs saved.\n")


# Does Restricting Inter-Event Time Attenuate the Recurrent-Disease Effect? (AoU)

Pooled, one-row-per-participant (latest transition). Three classifiers per gap cutoff (12/18/24 mo):
- **Any recurrence** (≥2 events, any gap)
- **Recent recurrence** (≥2 events AND gap ≤ cutoff)
- **Distant recurrence** (≥2 events AND gap > cutoff)

DeLong tests between them. Plus a subgroup test: among `first-event + distant-recurrence` only, does `≥2 events` still discriminate (AUC CI crossing 0.5 ⇒ effect attenuated).


In [ ]:
%%R

pooled_df_aou <- transitions_long %>%
  mutate(usable = case_when(
    had_next_event == 1 ~ TRUE,
    had_next_event == 0 & time_to_next_years >= follow_up_years ~ TRUE,
    TRUE ~ FALSE)) %>%
  filter(usable) %>%
  group_by(participant_id) %>%
  arrange(desc(from_event_num)) %>%
  slice(1) %>%
  ungroup() %>%
  mutate(actual = case_when(
    had_next_event == 1 ~ 1L,
    had_next_event == 0 & time_to_next_years >= follow_up_years ~ 0L,
    TRUE ~ NA_integer_)) %>%
  drop_na(actual)

cat("AoU pooled N =", nrow(pooled_df_aou), "\n")

gap_cutoffs <- c(12, 18, 24)

attenuate_metrics_list_aou <- list()
attenuate_delong_list_aou  <- list()
attenuate_roc_plots_aou    <- list()

auc_ci <- function(ro) {
  ci <- ci.auc(ro)
  c(auc = as.numeric(auc(ro)),
    lo  = as.numeric(ci[1]),
    hi  = as.numeric(ci[3]))
}

for (cutoff in gap_cutoffs) {
  df_c <- pooled_df_aou %>%
    mutate(
      any_rec     = as.integer(from_event_num >= 2),
      recent_rec  = as.integer(from_event_num >= 2 &
                               !is.na(prior_gap_months) &
                               prior_gap_months <= cutoff),
      distant_rec = as.integer(from_event_num >= 2 &
                               (is.na(prior_gap_months) |
                                prior_gap_months > cutoff))
    )

  ro_any     <- roc(df_c$actual, df_c$any_rec,     quiet = TRUE)
  ro_recent  <- roc(df_c$actual, df_c$recent_rec,  quiet = TRUE)
  ro_distant <- roc(df_c$actual, df_c$distant_rec, quiet = TRUE)

  s_any     <- auc_ci(ro_any)
  s_recent  <- auc_ci(ro_recent)
  s_distant <- auc_ci(ro_distant)

  rd_any     <- df_c %>% dplyr::rename(predicted = any_rec)
  rd_recent  <- df_c %>% dplyr::rename(predicted = recent_rec)
  rd_distant <- df_c %>% dplyr::rename(predicted = distant_rec)

  m_any     <- extract_metrics_grid(ro_any,
                 "\u22651 subsequent stone event", rd_any) %>%
    mutate(Gap_Cutoff_Months = cutoff,
           Classifier = "Any recurrence (\u22652 events)")
  m_recent  <- extract_metrics_grid(ro_recent,
                 "\u22651 subsequent stone event", rd_recent) %>%
    mutate(Gap_Cutoff_Months = cutoff,
           Classifier = paste0("Recent recurrence (gap \u2264", cutoff, "mo)"))
  m_distant <- extract_metrics_grid(ro_distant,
                 "\u22651 subsequent stone event", rd_distant) %>%
    mutate(Gap_Cutoff_Months = cutoff,
           Classifier = paste0("Distant recurrence (gap >", cutoff, "mo)"))

  attenuate_metrics_list_aou[[paste0("cut_", cutoff)]] <-
    bind_rows(m_any, m_recent, m_distant)

  dl_any_vs_recent  <- tryCatch(roc.test(ro_any, ro_recent,  method = "delong"),
                                error = function(e) NULL)
  dl_any_vs_distant <- tryCatch(roc.test(ro_any, ro_distant, method = "delong"),
                                error = function(e) NULL)
  dl_recent_vs_dist <- tryCatch(roc.test(ro_recent, ro_distant, method = "delong"),
                                error = function(e) NULL)

  attenuate_delong_list_aou[[paste0("cut_", cutoff)]] <- data.frame(
    Gap_Cutoff_Months = cutoff,
    Comparison = c("Any vs Recent","Any vs Distant","Recent vs Distant"),
    delong_p = c(
      if (is.null(dl_any_vs_recent))  NA_real_ else as.numeric(dl_any_vs_recent$p.value),
      if (is.null(dl_any_vs_distant)) NA_real_ else as.numeric(dl_any_vs_distant$p.value),
      if (is.null(dl_recent_vs_dist)) NA_real_ else as.numeric(dl_recent_vs_dist$p.value)
    )
  )

  cat(sprintf(
    "\n--- AoU Gap cutoff: %dmo ---\n  Any    AUC = %.3f (%.3f, %.3f)\n  Recent AUC = %.3f (%.3f, %.3f)\n  Distant AUC = %.3f (%.3f, %.3f)\n  DeLong Any vs Recent:  p = %s\n  DeLong Any vs Distant: p = %s\n  DeLong Recent vs Distant: p = %s\n",
    cutoff,
    s_any["auc"], s_any["lo"], s_any["hi"],
    s_recent["auc"], s_recent["lo"], s_recent["hi"],
    s_distant["auc"], s_distant["lo"], s_distant["hi"],
    ifelse(is.null(dl_any_vs_recent),  "NA", format.pval(dl_any_vs_recent$p.value,  digits = 3)),
    ifelse(is.null(dl_any_vs_distant), "NA", format.pval(dl_any_vs_distant$p.value, digits = 3)),
    ifelse(is.null(dl_recent_vs_dist), "NA", format.pval(dl_recent_vs_dist$p.value, digits = 3))
  ))

  roc_ov <- rbind(
    data.frame(fpr = 1 - ro_any$specificities,     tpr = ro_any$sensitivities,
               Classifier = paste0("Any recurrence (AUC=", round(s_any["auc"],3), ")")),
    data.frame(fpr = 1 - ro_recent$specificities,  tpr = ro_recent$sensitivities,
               Classifier = paste0("Recent \u2264", cutoff, "mo (AUC=", round(s_recent["auc"],3), ")")),
    data.frame(fpr = 1 - ro_distant$specificities, tpr = ro_distant$sensitivities,
               Classifier = paste0("Distant >", cutoff, "mo (AUC=", round(s_distant["auc"],3), ")"))
  )

  p_roc <- ggplot(roc_ov, aes(x = fpr, y = tpr, colour = Classifier)) +
    geom_line(linewidth = 1) +
    geom_abline(linetype = "dashed", alpha = 0.5) +
    coord_equal() +
    labs(x = "False Positive Rate", y = "True Positive Rate",
         title = paste0("AoU: attenuation test, gap cutoff ", cutoff, "mo"),
         subtitle = paste0("Pooled N = ", nrow(pooled_df_aou)),
         colour = "Classifier") +
    scale_colour_manual(values = c("#1b9e77","#d95f02","#7570b3")) +
    theme_bw(base_size = 12) +
    theme(legend.position = "bottom") +
    guides(colour = guide_legend(nrow = 3))

  attenuate_roc_plots_aou[[paste0("cut_", cutoff)]] <- p_roc
  print(p_roc)

  ggsave(paste0("attenuation_test_roc_cut_", cutoff, "_aou.pdf"),
         plot = p_roc, width = 9, height = 7)
}

attenuate_metrics_df_aou <- bind_rows(attenuate_metrics_list_aou)
attenuate_delong_df_aou  <- bind_rows(attenuate_delong_list_aou) %>%
  mutate(delong_p_fmt = ifelse(is.na(delong_p), NA,
                               ifelse(delong_p < 0.001, "<0.001",
                                      sprintf("%.3f", delong_p))))

write.csv(attenuate_metrics_df_aou,
          "attenuation_test_auc_metrics_aou.csv", row.names = FALSE)
write.csv(attenuate_delong_df_aou,
          "attenuation_test_delong_aou.csv", row.names = FALSE)

print(attenuate_metrics_df_aou %>%
  dplyr::select(Gap_Cutoff_Months, Classifier,
                N, N_High_Risk, N_Low_Risk,
                AUC, Sensitivity, Specificity, PPV, NPV) %>%
  arrange(Gap_Cutoff_Months, Classifier))

print(attenuate_delong_df_aou %>%
  dplyr::select(Gap_Cutoff_Months, Comparison, delong_p_fmt))


## Subgroup Test (AoU): Does Event-Count Still Discriminate Among Long-Gap + First-Event Cases?


In [ ]:
%%R

attenuate_subgroup_list_aou <- list()

for (cutoff in gap_cutoffs) {
  sub_df <- pooled_df_aou %>%
    filter(from_event_num == 1 |
           (from_event_num >= 2 & (is.na(prior_gap_months) | prior_gap_months > cutoff))) %>%
    mutate(predicted = as.integer(from_event_num >= 2))

  if (length(unique(sub_df$predicted)) < 2 ||
      length(unique(sub_df$actual))    < 2) next

  ro_sub <- roc(sub_df$actual, sub_df$predicted, quiet = TRUE)
  ci_sub <- ci.auc(ro_sub)

  n_boot <- 1000
  set.seed(42)
  boot_aucs <- replicate(n_boot, {
    idx <- sample(seq_len(nrow(sub_df)), replace = TRUE)
    boot_df <- sub_df[idx, ]
    if (length(unique(boot_df$actual)) < 2) return(NA_real_)
    as.numeric(auc(roc(boot_df$actual, boot_df$predicted, quiet = TRUE)))
  })
  boot_aucs <- boot_aucs[!is.na(boot_aucs)]
  p_vs_half <- 2 * min(mean(boot_aucs <= 0.5), mean(boot_aucs >= 0.5))

  m_sub <- extract_metrics_grid(ro_sub,
    "\u22651 subsequent stone event", sub_df) %>%
    mutate(Gap_Cutoff_Months = cutoff,
           N_total = nrow(sub_df),
           boot_p_vs_0.5 = p_vs_half)

  attenuate_subgroup_list_aou[[paste0("cut_", cutoff)]] <- m_sub

  cat(sprintf(
    "\nAoU Subgroup gap > %dmo: N=%d  AUC=%.3f (%.3f, %.3f)  boot p vs 0.5 = %.3f\n",
    cutoff, nrow(sub_df),
    as.numeric(auc(ro_sub)), ci_sub[1], ci_sub[3], p_vs_half))
}

attenuate_subgroup_df_aou <- bind_rows(attenuate_subgroup_list_aou)

write.csv(attenuate_subgroup_df_aou,
          "attenuation_test_subgroup_aou.csv", row.names = FALSE)

print(attenuate_subgroup_df_aou %>%
  dplyr::select(Gap_Cutoff_Months, N_total, N_High_Risk, N_Low_Risk,
                AUC, boot_p_vs_0.5, Sensitivity, Specificity, PPV, NPV))

cat("AoU attenuation-test outputs saved.\n")
